> **Related notebook — `claude_code_from_scratch.ipynb`.** That notebook is the complement to this one. Where this notebook surveys the *62 reasoning and reliability patterns inside the model* (cloud DeepSeek API, breadth-first, 8 phases), `claude_code_from_scratch.ipynb` builds *one runnable Claude-Code-style coding-agent harness* — perception-action loop, file/shell tools, subagent isolation, context compaction, task DAG — against **local Ollama (qwen3)**. Read this notebook for the *what/why* of model cognition; read that one for the *how* of a working coding agent.

# Building Claude from Scratch: 62 Components Behind Anthropic's Thinking Engine

> A runnable notebook reproduction of Fareed Khan's article
> *"Building Claude from Scratch: 62 Components Behind Anthropic's Thinking Engine"*
> (Level Up Coding, May 2026).
> Original code + theory: https://github.com/FareedKhan-dev/building-claude-from-scratch

---

**What this notebook is.** It walks through all 8 phases of building a
Claude-like agentic *harness* on top of an open-source model (DeepSeek), and
applies it to a single hard task: reproducing the national 75th-percentile
dengue forecast (1,405,191 cases) from Freitas et al. 2025 within 5%.

**How to run it.**

1. Set an API key for an OpenAI-compatible endpoint:
   ```bash
   export DEEPSEEK_API_KEY="sk-..."
   ```
   The same code works with any OpenAI-compatible backend (vLLM, Ollama,
   OpenRouter, Together, etc.) by changing `base_url` / `MODEL_*`.
2. Provide the workspace data the later phases expect under
   `./seird_workspace/` (`paper.txt` and `data/cases.csv.gz` — the DATASUS
   dengue surveillance dataset). Phases 1-6 that only call the LLM run without
   the dataset; Phases 7-8 additionally need Docker for the sandbox.

**About the outputs.** Each code cell is the real code from the article.
Where the article showed a result, it is reproduced here in an
`Output (from the article)` block so you can follow along without spending
API budget. Re-running the cells against a live model will produce similar
(not byte-identical) results.

**Note:** one copy/paste indentation slip in the article's first
`think_then_answer` cell has been corrected so the cell is valid Python.


# Building Claude from Scratch: 62 Components Behind Anthropic’s Thinking Engine

## Machine learning patterns and agentic components, end to end

In practice, when building agentic systems, AI models are rarely the bottleneck anymore. The harness around them is. Anthropic spent two years building that harness for Claude, the orchestration code that picks the right tools and grades its work before declaring success. Claude itself is built around 62 carefully composed components spanning machine learning patterns like compute optimal allocation, deliberative alignment, bi temporal memory, alongside agentic patterns like the OODA loop, plan and execute, architect editor splits, and many others.

*Claude Like Thinking (Created by Fareed Khan)*

Those 62 components that define Claude’s thinking approach are distributed across 4 main principles.

- Cognition: how the model deliberates before it acts. Thinking channels, compute adaptive allocation, self consistency, and best of N verification turn a chat completion into a reasoner.

- Orchestration: how thoughts compose into trajectories. Tree of thoughts, OODA subagents, plan and execute, and the master loop decompose hard tasks into solvable subgoals.

- Reliability: how the harness prevents and detects failures. Architect editor splits, linter in the loop, self refine, and cache aware prompts turn a working agent into a deployable one.

- Grounding and Trust: how the agent earns the right to claim success. Persistent sandboxed REPLs, real pytest verification, four tier memory, and definition of done contracts make the verdict mechanically auditable.

In this blog we are going to technically replicate Claude’s behavior on an open source model and see how Anthropic engineered each layer by looking into their papers, and in parallel build all 62 components to solve a complex multi-step problem that people would normally solve through Claude.

All the code (Notebook) + theory is available in my GitHub repository:

## Table of Contents

- The Problem We Are Solving

- Setting Up the Foundation

- Phase 1 — The Cognitive Substrate: Teaching the Model to Deliberate ∘ The Thinking Channel and Interleaved Reasoning ∘ Compute-Adaptive Effort Allocation ∘ Search-Based Decoding: Self-Consistency, Best-of-N, Budget Forcing ∘ Bare-Model Baseline vs Thinking-Enabled Baseline

- Phase 2 — Reasoning Topology: Decomposition, Branching, and Sub-Agent Discipline ∘ Step-Back Abstraction and Least-to-Most Decomposition ∘ Tree of Thoughts and the OODA Subagent Pattern ∘ The Master Loop with Orchestrator-Worker Topology ∘ Sub-Agent Output Discipline

- Phase 3 — Tool-Grounded Execution: From Plan to Verifiable Action ∘ Plan-and-Execute and the LLM Compiler ∘ ReAct, Evaluator-Optimizer, Reflexion: The Self-Correction Family ∘ CRITIC and Mixture-of-Agents ∘ Verifier Asymmetry

- Phase 4 — Production Reliability: The Hardening Stack ∘ Self Refine, Verifier Guided Search, External Feedback Verification ∘ Tool Description Self Improvement and Adversarial Self Probing ∘ Editorial Discipline Triad: Architect Editor, Linter, Result Compaction ∘ Cache Aware Prompt Ordering, Sample Diversity, Anti Counting Pattern

- Phase 5 — Frontier Only Patterns: What Makes Claude Feel Different ∘ Thought Signatures, Goldilocks Altitude, and Token Variance ∘ Compute Optimal Allocation, Coverage Curves, Soul Document ∘ Deliberative Alignment and the Effort Knob ∘ Delegation Cost, Strict Tool Choice, Process Isolation, Bi Temporal Memory

- Phase 6 — Meta Cognition and Stateful Orchestration ∘ Problem Type Classification and Cost Bounded Branching ∘ Execution Trace as First Class Artifact and Definition of Done Contract ∘ The Persistent Task DAG with SQLite Backed State ∘ Selective Rollback and Replan as Graph Mutation

- Phase 7 — Grounding, Evaluation, and the Trust Gate ∘ The Persistent Sandboxed REPL and Filesystem as State ∘ Real Environment Verification and the Executable Spec Layer ∘ The Four Tier Memory System ∘ The MCP Compatible Tool Registry

- Phase 8 — Composition and the End to End Reproduction Run ∘ The Five Subagent Architecture ∘ The Master Loop and the Reproduction Run ∘ Understanding Results

- Summarizing the Architecture

## The Problem We Are Solving

Before we start building, we need a real problem that actual devs, engineers, and researchers solve through Claude. Only then can we evaluate the results in a meaningful way.

Normally a real world problem is a multi step problem where the model needs to remember each line it produces, because every step deals with numbers, outputs, file states, and decisions that depend on the previous one. A single forgotten variable or a single wrong assumption breaks the whole chain.

So we picked Freitas et al. 2025, “A statistical model for forecasting probabilistic epidemic bands for dengue cases in Brazil” (Infectious Disease Modelling, doi:10.1016/j.idm.2025.07.014).

- The paper fits a Bayesian hierarchical model on 14 years of DATASUS dengue data and forecasts the 2022 to 2023 epidemic season.

- Their reported national 75th percentile estimate is 1,405,191 cases. Our agent’s job is to reproduce this number within 5 percent on its own.

Why this paper. It has clean math, a public dataset, a single hard numerical target, and a real Bayesian inference step. It is exactly the kind of task a researcher would normally hand to Claude.

> Read the paper. Write the code. Fit the model. Validate the result. Write the report.

Paper: https://doi.org/10.1016/j.idm.2025.07.014

Dataset: DATASUS public dengue surveillance data, 487,239 weekly observations across 5,570 municipalities, covering 2010 to 2024.

## Setting Up the Foundation

Before we build any phase, we need the shared foundation: a client connection to the open-source model, a base system prompt, and the dataset.

In [ ]:
# core.py — shared foundation for every phase
import os
from openai import OpenAI

# OpenAI-compatible client pointed at DeepSeek's hosted API
# Same code works with any OpenAI-compatible endpoint:
# vLLM (local), Ollama, OpenRouter, Together, Anthropic-via-litellm
client = OpenAI(
    api_key   = os.environ["DEEPSEEK_API_KEY"],
    base_url  = "https://api.deepseek.com/v1",
)

MODEL_FAST      = "deepseek-chat"      # for routine work
MODEL_REASONING = "deepseek-reasoner"  # for hard reasoning steps

print(f"Connected to DeepSeek API")
print(f"  fast model:      {MODEL_FAST}")
print(f"  reasoning model: {MODEL_REASONING}")

**Output (from the article):**

```text
Connected to DeepSeek API
  fast model:      deepseek-chat
  reasoning model: deepseek-reasoner
```

Two model tiers is the same pattern Claude uses internally. Anthropic exposes Claude as Haiku, Sonnet, and Opus precisely so users can build the architect/editor split (Phase 4) where the strong model designs and the cheap model implements. We are doing the structural equivalent with DeepSeek’s two tiers.

The base system prompt is the foundation of the agent’s identity:

In [ ]:
STRONG_SYSTEM_PROMPT = (
    "You are a careful, senior research-engineer agent. Your job is to "
    "reproduce a peer-reviewed scientific paper end-to-end. You think "
    "before you act. You write code that is verifiable, not impressive. "
    "You name your assumptions before you commit to them.\n\n"

    "RULES OF ENGAGEMENT:\n"
    "1. Never claim a result without a runnable artefact backing it.\n"
    "2. Defer all numerical questions to your code execution tool.\n"
    "3. When a verifier disagrees with you, the verifier is correct\n"
    "   until you produce evidence to the contrary.\n"
    "4. If you do not know how to do something, say so. Do not guess.\n"
    "5. The contract is the source of truth. Your opinion of your own\n"
    "   work does not override the spec layer's verdict.\n"
)

print(f"System prompt: {len(STRONG_SYSTEM_PROMPT)} chars")

**Output (from the article):**

```text
System prompt: 928 chars
```

> Notice rules 1, 2, and 5. They map directly to Anthropic’s published Claude design principles, no claims without artefacts, defer numerical questions to code, the verifier overrides the model’s opinion of itself.

These are the rules that prevent the model from confidently fabricating numbers, which is the single most common failure mode of bare LLMs on research tasks.

Now we load the actual paper and dataset:

In [ ]:
import pandas as pd
from pathlib import Path


WORKSPACE      = Path("./seird_workspace")
WORKSPACE.mkdir(exist_ok=True)
AGENT_CODE_DIR = WORKSPACE / "agent_code"
AGENT_CODE_DIR.mkdir(exist_ok=True)

# Load the paper text (extracted from PDF in Part 2 of the notebook)
paper_text = (WORKSPACE / "paper.txt").read_text()
print(f"Paper loaded:        {len(paper_text):,} chars")

# Load the real DATASUS dengue dataset - 487,239 rows of weekly municipal cases
cases = pd.read_csv(WORKSPACE / "data" / "cases.csv.gz")

print(f"Dataset shape:       {cases.shape}")
print(f"Date range:          {cases['data_iniSE'].min()} to {cases['data_iniSE'].max()}")
print(f"Unique municipalities: {cases['municipio_geocodigo'].nunique()}")
print(f"Total probable cases: {int(cases['casos_prov'].sum()):,}")

**Output (from the article):**

```text
Paper loaded:        64,213 chars
Dataset shape:       (487239, 5)
Date range:          2010-01-03 to 2024-09-29
Unique municipalities: 5570
Total probable cases: 13,194,022
```

5,570 municipalities matches the paper’s reported municipality count exactly. 487,239 weekly observations across 14 years of DATASUS data. The 13.2 million total cases is the number the agent will validate against later.

The paper’s specific reproduction target: filter to the 2022–2023 dengue season (October 2022 through October 2023) and the total observed cases should be 1,436,034. The paper’s BYM2+RW1 model produces a national 75th-percentile posterior estimate of 1,405,191 for that same season. Our agent’s job is to reproduce this number within 5%.

In [ ]:
# The paper's reproduction target
season = cases[(cases['data_iniSE'] >= '2022-10-09') &
               (cases['data_iniSE'] <= '2023-10-01')]
observed_total = int(season['casos_prov'].sum())

print(f"2022-2023 season observed total: {observed_total:,}")
print(f"Paper's reported observed total: 1,436,034")
print(f"Match: {observed_total == 1_436_034}")

**Output (from the article):**

```text
2022-2023 season observed total: 1,436,034
Papers reported observed total: 1,436,034
Match: True
```

The dataset matches the paper to the digit. We now have the ground truth that everything downstream will be measured against.

## Phase 1 — The Cognitive Substrate: Teaching the Model to Deliberate

A bare LLM is a chat completion. You give it a prompt, it returns text. There is no thinking, no checking, no verification. For research paper reproduction this is catastrophic that the model will confidently produce prose about how you would reproduce the paper while never producing a single runnable file. Phase 1 builds the cognitive substrate that turns a chat completion into a reasoner.

> Claude’s extended thinking mode is the canonical example of this layer.

*Phase 1 (Teaching the Model) Created by Fareed Khan*

When Claude is asked a hard question, it produces a <thinking> block before its answer. The thinking block contains its actual reasoning, the answer block contains the user-facing output. We will replicate this structure on the open-source model.

### The Thinking Channel and Interleaved Reasoning

This is technique #1 and #2 of the 62. The thinking channel separates what the model is reasoning about from what it tells the user. Interleaved reasoning extends this by re-running the thinking block between tool calls, so the model is constantly re-evaluating its plan based on new observations.

*Thinking Channel (Created by Fareed Khan)*

The change is structural. We do not just prepend “think step by step” to the prompt. We add a system instruction that requires the model to emit two distinct blocks one for thinking, one for the answer and we parse them separately downstream.

In [ ]:
THINKING_SYSTEM_PROMPT = (
    "Produce your response in two parts:\n"
    "  <thinking>\n"
    "  Step through the question carefully. Decompose. Consider the\n"
    "  options. Identify the most likely failure mode of a quick answer.\n"
    "  Be honest about uncertainty.\n"
    "  </thinking>\n"
    "  <answer>\n"
    "  The actual answer, concise and direct. No hedging unless the\n"
    "  uncertainty is genuine.\n"
    "  </answer>\n"
    "Always emit BOTH tags. Never omit the thinking block."
)

The function that wraps this is short:

In [ ]:
import re
from dataclasses import dataclass

@dataclass
class ThoughtfulResponse:
    thinking: str
    answer: str
    output_tokens: int


def think_then_answer(query: str, model: str = MODEL_FAST,
                      max_tokens: int = 800, temperature: float = 0.3) -> ThoughtfulResponse:
    """Force the model to emit a <thinking> block before its <answer>."""
    resp = client.chat.completions.create(
        model = model,
        messages = [
            {"role": "system", "content": STRONG_SYSTEM_PROMPT + "\n\n" + THINKING_SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ],
        temperature = temperature,
        max_tokens  = max_tokens,
    )
    raw = resp.choices[0].message.content
    thinking = re.search(r"<thinking>(.*?)</thinking>", raw, re.DOTALL)
    answer   = re.search(r"<answer>(.*?)</answer>",     raw, re.DOTALL)
    return ThoughtfulResponse(
        thinking      = thinking.group(1).strip() if thinking else "",
        answer        = answer.group(1).strip()   if answer   else raw.strip(),
        output_tokens = resp.usage.completion_tokens,
    )

Now we run it on a real question from our paper-reproduction context:

In [ ]:
result = think_then_answer(
    "The Freitas 2025 paper uses BYM2 priors for the spatial random effect. "
    "What does the BYM2 reparameterisation buy us over a plain CAR prior, "
    "and why does it matter for the dengue forecasting case specifically?"
)

print("THINKING BLOCK:")
print(result.thinking)

print()

print("ANSWER BLOCK:")
print(result.answer)

print()

print(f"Output tokens: {result.output_tokens}")

**Output (from the article):**

```text
THINKING BLOCK:
The user is asking about a Bayesian spatial modelling choice. Let me decompose:
1. What is BYM2 specifically? It's the Riebler et al. 2016 reparameterisation
   that splits the spatial random effect into a structured (CAR) and
   unstructured (iid) component, with a mixing parameter phi in [0,1].
2. What does plain CAR lack? CAR doesn't separate structured spatial smoothing
   from independent district-level noise. The hyperparameters are not
   intuitively interpretable — variance has no fixed scale.
3. Why does BYM2 matter for dengue specifically? Brazilian municipalities
   have wildly different population sizes. The structured CAR component
   captures spatial diffusion of dengue transmission; the iid component
   captures unexplained municipality-level variation (local mosquito
   populations, surveillance quality differences). The phi parameter then
   tells us how much of the variance is spatial vs municipal-specific —
   directly interpretable.
4. Failure mode of a quick answer: just saying "BYM2 is better than CAR"
   without naming the specific Riebler scaling and the interpretable phi.


ANSWER BLOCK:
BYM2 (Riebler et al. 2016) decomposes the spatial random effect into a
structured CAR component and an unstructured iid component, with a mixing
parameter phi in [0,1] indicating how much of the variance is spatial. It
also rescales the precision so the prior is comparable across different
graph topologies - plain CAR is not. For Brazilian dengue specifically,
BYM2 lets us read directly from the posterior how much of the
between-municipality variation is explained by spatial diffusion versus
municipality-specific factors.

Output tokens: 487
```

The model’s thinking block does what we asked. It decomposes the question, names the specific paper (Riebler 2016), names the specific parameter (phi in [0,1]), and explicitly considers the failure mode of giving a shallow answer. The answer block is what the user sees concise, technically correct, no padding.

This pattern, forced separation of reasoning from output, is what Claude does internally on every query that uses extended thinking. Anthropic’s published architecture documentation calls this the “thought channel”. We just reproduced its structural shape on the open-source model.

### Compute-Adaptive Effort Allocation

In this section we are going to cover techniques #3, #4, and #5, static budgets, adaptive budgets, and test-time compute scaling. The core insight: not every question deserves the same amount of compute. A trivial question should consume 100 thinking tokens, a hard structural question should consume 2,000.

*Compute Adaptive (Created by Fareed Khan)*

Claude exposes this directly through the thinking_budget parameter in its API. Open-source models do not have that knob, but we can replicate it by allocating different max_tokens budgets based on the question's classified difficulty.

In [ ]:
def estimate_difficulty(query: str) -> str:
    """Cheap classifier — uses fast model with low max_tokens."""
    resp = client.chat.completions.create(
        model = MODEL_FAST,
        messages = [
            {"role": "system", "content":
             "Classify the difficulty of this question. Output ONLY one word: "
             "trivial, easy, medium, hard, or extreme."},
            {"role": "user", "content": query},
        ],
        max_tokens  = 5,
        temperature = 0.0,
    )
    return resp.choices[0].message.content.strip().lower()


# Map difficulty to thinking budget
THINKING_BUDGETS = {
    "trivial": 100,
    "easy":    300,
    "medium":  800,
    "hard":    2000,
    "extreme": 4000,
}

def adaptive_think(query: str) -> dict:
    difficulty = estimate_difficulty(query)
    budget     = THINKING_BUDGETS.get(difficulty, 800)
    print(f"  classified: {difficulty}  → budget: {budget} tokens")
    
    response = think_then_answer(query, max_tokens=budget)
    return {
        "difficulty": difficulty,
        "budget":     budget,
        "actual_tokens": response.output_tokens,
        "answer":     response.answer,
    }

We run this on three questions of escalating difficulty:

In [ ]:
questions = [
    "What library does Python use for ODE integration?",
    "How would you set up a BYM2 spatial random effect in INLA?",
    ("Given the dengue dataset has 5,570 municipalities but only 118 health districts, "
     "explain why aggregating to district level changes the posterior variance "
     "structure of the spatial random effect, and what tradeoff this implies."),
]

for q in questions:
    print(f"Question: {q[:80]}...")
    result = adaptive_think(q)
    print(f"  actual tokens used: {result['actual_tokens']}")
    print()

**Output (from the article):**

```text
Question: What library does Python use for ODE integration?...
  classified: trivial  → budget: 100 tokens
  actual tokens used: 67

Question: How would you set up a BYM2 spatial random effect in INLA?...
  classified: medium  → budget: 800 tokens
  actual tokens used: 743

Question: Given the dengue dataset has 5,570 municipalities but only 118 health districts...
  classified: hard  → budget: 2000 tokens
  actual tokens used: 1842
```

The classifier picks the right band each time, and the model adapts its thinking depth accordingly. The trivial question used 67 tokens. The hard question used 1,842 tokens, 27× more compute on a 27× harder problem. Without the classifier, every question would have used the same fixed budget, either underspending on hard questions or overspending on trivial ones.

This is what Anthropic means by compute-optimal allocation:

> the harness decides how much compute to spend, not the user. Claude does this internally, we just made it explicit at the inference layer.

### Search-Based Decoding: Self-Consistency, Best-of-N, and Budget Forcing

Techniques #6, #7, and #8. The shared idea: a single sample from the model is a single point estimate of the answer. K samples gives you a distribution. The right answer often appears in the distribution but not in any single sample.

> Self-consistency (Wang et al. 2022) generates K samples and takes the majority vote. It works for questions with discrete answers.

*Search-based (Created by Fareed Khan)*

Best-of-N generates K samples and uses a separate verifier to pick the best one. It works for any task where you can score outputs.

Budget forcing (Snell et al. 2024) appends “Wait, let me reconsider” to the model’s output mid-generation, forcing it to extend its reasoning when the first attempt looks weak.

We will build all three. Self-consistency first:

In [ ]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor


def self_consistency(query: str, k: int = 5, model: str = MODEL_FAST) -> dict:
    """Sample k answers in parallel, return the majority vote."""
    def _one(_):
        resp = think_then_answer(query, model=model, temperature=0.7)
        return resp.answer.strip()
    with ThreadPoolExecutor(max_workers=k) as ex:
        samples = list(ex.map(_one, range(k)))

    # Take the majority answer (handles slight wording variations
    # by normalising - first 50 chars as the bucket key)
    keys     = [s[:50].lower() for s in samples]
    counter  = Counter(keys)
    winner_key, votes = counter.most_common(1)[0]
    winner   = next(s for s in samples if s[:50].lower() == winner_key)

    return {
        "winner":     winner,
        "votes":      votes,
        "k":          k,
        "agreement":  votes / k,
        "all_samples": samples,
    }

We test it on a question with a discrete numerical answer:

In [ ]:
result = self_consistency(
    "How many spatial random-effect components are there in a BYM2 specification "
    "(spatial + unstructured + total mixing parameter)? Answer with just the number.",
    k = 5,
)
print(f"Winner ({result['votes']}/{result['k']} agreement): {result['winner']}")
print(f"All samples: {[s[:30] for s in result['all_samples']]}")

**Output (from the article):**

```text
Winner (4/5 agreement): 3 (one structured CAR component, one unstructured iid component, and one mixing parameter phi).
All samples: ['3 (one structured CAR component',
              '3 (structured + unstructured + ',
              '3 — the structured CAR, the un',
              '2 (structured and unstructured)',
              '3 (CAR + iid + phi mixing).']
```

Five samples, four said 3, one said 2. Self-consistency picked 3 with 80% agreement. Without self-consistency, a single sample at temperature 0.7 had a 20% chance of returning the wrong answer. With self-consistency, we get the right answer 4 times out of 5 in the vote.

Best-of-N uses a separate verifier instead of voting. The verifier is the strong model; the generator is the fast model. This is asymmetric — it costs more on the verifier side, but the verifier is called once on K candidates instead of K times:

In [ ]:
import json

VERIFIER_SYSTEM = (
    "You are a careful, structured verifier. You score the candidate "
    "answer on a 1-10 scale where 10 is perfect and 1 is unusable. "
    "Your score must reflect FACTS, not style. Output JSON: "
    '{"score": int (1-10), "reason": str (one sentence)}.'
)

def verifier_score(question: str, candidate: str,
                    verifier_model: str = MODEL_REASONING) -> dict:
    resp = client.chat.completions.create(
        model = verifier_model,
        messages = [
            {"role": "system", "content": VERIFIER_SYSTEM},
            {"role": "user",   "content": f"QUESTION:\n{question}\n\nCANDIDATE:\n{candidate}"},
        ],
        response_format = {"type": "json_object"},
        temperature = 0.0,
        max_tokens  = 200,
    )
    return json.loads(resp.choices[0].message.content)

def best_of_n(query: str, n: int = 4) -> dict:
    """Generate n candidates with the fast model, pick the best per the strong-model verifier."""
    with ThreadPoolExecutor(max_workers=n) as ex:
        candidates = list(ex.map(
            lambda _: think_then_answer(query, temperature=0.7).answer,
            range(n),
        ))
    scored = [{"answer": c, **verifier_score(query, c)} for c in candidates]
    scored.sort(key=lambda x: x["score"], reverse=True)
    return {"winner": scored[0], "all": scored}

Run it on a tricky question:

In [ ]:
result = best_of_n(
    "What is the difference between INLA and PyMC for fitting hierarchical "
    "Bayesian models, and which one would be the right choice for the Freitas "
    "dengue paper?",
    n = 4,
)

print(f"WINNER (score {result['winner']['score']}/10):")
print(f"  reason: {result['winner']['reason']}")

print()

print("All candidates ranked:")

for c in result["all"]:
    print(f"  {c['score']}/10 - {c['reason']}")

**Output (from the article):**

```text
WINNER (score 9/10):
  reason: Correctly identifies INLA as the paper's choice via integrated nested Laplace approximation, contrasts with PyMC's NUTS sampling, and gives the right tradeoff (speed vs flexibility).

All candidates ranked:
  9/10 - Correctly identifies INLA as the paper's choice via integrated nested Laplace approximation, contrasts with PyMC's NUTS sampling, and gives the right tradeoff (speed vs flexibility).
  7/10 - Mentions INLA and PyMC but does not explain the I-N-L-A initialism precisely; tradeoff section is vague.
  6/10 - Conflates INLA with simple Laplace approximation; misses the integrated/nested distinction.
  4/10 - Recommends PyMC for the Freitas paper, which contradicts the paper's actual choice.
```

The fast model produced four candidates of widely different quality. The strong-model verifier correctly identified the 9/10 candidate. Without the verifier, we would have committed to the median of the four quality 6 or 7. With the verifier, we get the 9.

This is exactly how Claude Code’s internal “consider alternatives” mode works on hard design questions.

### Bare-Model Baseline vs Thinking-Enabled Baseline

To measure what the cognitive substrate actually buys us, we need a baseline. We give the bare model the same task we will eventually give the full agent — “reproduce the Freitas 2025 dengue paper end-to-end” with no thinking, no tools, no anything. Just the model.

*Comparison (Created by Fareed Khan)*

In [ ]:
def bare_model(query: str) -> str:
    """Model with no harness. Direct API call."""
    resp = client.chat.completions.create(
        model    = MODEL_FAST,
        messages = [{"role": "user", "content": query}],
        max_tokens = 800,
    )
    return resp.choices[0].message.content

bare_response = bare_model(
    "Reproduce the Freitas et al. 2025 dengue-forecasting paper end-to-end. "
    "Fit the BYM2+RW1 model on the 12 training seasons, forecast 2022-2023, "
    "and verify the national 75th-percentile estimate is within 5% of "
    "1,405,191 cases."
)

print(bare_response[:600])

**Output (from the article):**

```text
To reproduce the Freitas et al. 2025 dengue paper, you would
fit a hierarchical Bayesian model to weekly case counts across
Brazilian health districts using something like INLA or Stan.
The 75th-percentile forecast for the 2022-2023 season is around
1.4 million cases, which matches the order of magnitude they
report.

You'll want to use BYM2 priors for spatial effects and
AR1 or RW1 for temporal. Once fit, validate against the held-out
season to check forecast skill.
```

The bare model’s output is useless. It mentions BYM2, RW1, INLA all correct surface features. But it is prose about a method, not a method that ran. “Around 1.4 million” is not a posterior estimate, it is a guess. The number is also not within 5% of 1,405,191 because there is no number to check.

This is the baseline. Through 8 phases of harness engineering, we are going to turn this prose into five real Python files on disk, a real Bayesian model fit on the real dataset, a real posterior sample, and a real verdict graded by a real spec layer.

> The cost of that bare-model output: $0.000041, 3.2 seconds. Remember those numbers. We will compare them against the full harness in Phase 8.

Phase 2 takes these atomic reasoning capabilities and composes them into trajectories, how thoughts decompose into subgoals, branch into trees, and propagate through subagent hierarchies.

## Phase 2 — Reasoning Topology: Decomposition, Branching, and Sub-Agent Discipline

Phase 1 gave the model the ability to deliberate. But deliberation alone is not enough for paper reproduction. The Freitas paper has at minimum 8 distinct subgoals: load data, aggregate to district-week, build the spatial adjacency, specify the model, fit it, compute the posterior 75th percentile, validate against the paper, write a report.

*Phase 2 (reasoning topology) Created by Fareed Khan*

A single “think and answer” cycle cannot produce all of this. The model needs to decompose the problem, branch across alternatives where the right approach is unclear, and delegate isolated subtasks to sub-agents whose intermediate work never pollutes the main reasoning context.

This is the reasoning topology layer. Claude does this through its master loop and parallel subagent spawning. We will replicate the structural pattern.

### Step-Back Abstraction and Least-to-Most Decomposition

Techniques #9 and #10. Step-back prompting (Zheng et al. 2023) tells the model “before answering the specific question, first state the general principle that applies”. Least-to-most decomposition (Zhou et al. 2022) tells the model “break this into the smallest sub-questions where each one’s answer feeds the next”.

> Both are anti-failure-mode patterns they prevent the model from over-fitting on surface details and missing the structural shape of the problem.

*Step-back abstraction (Created by Fareed Khan)*

Anthropic published examples for Claude show this pattern in their “plan before you code” discipline. When Claude is asked to refactor a large file, it first states the general principle of the refactor before writing any code. We make this explicit:

In [ ]:
def step_back_then_solve(query: str, model: str = MODEL_REASONING) -> dict:
    """First ask for the general principle, then apply it to the specific question."""
    
    # Step 1: extract the general principle
    principle_resp = client.chat.completions.create(
        model    = model,
        messages = [
            {"role": "system", "content":
             "Before answering the specific question, state the GENERAL PRINCIPLE "
             "or pattern that applies to questions of this category. One paragraph. "
             "Do not answer the specific question yet."},
            {"role": "user", "content": query},
        ],
        max_tokens = 300,
    )
    principle = principle_resp.choices[0].message.content
    
    # Step 2: apply the principle to the specific question
    solve_resp = client.chat.completions.create(
        model    = model,
        messages = [
            {"role": "system", "content": STRONG_SYSTEM_PROMPT},
            {"role": "user",   "content": f"GENERAL PRINCIPLE:\n{principle}\n\nSPECIFIC QUESTION:\n{query}"},
        ],
        max_tokens = 800,
    )
    answer = solve_resp.choices[0].message.content
    
    return {"principle": principle, "answer": answer}

We test this on a tricky modelling question from the paper:

In [ ]:
result = step_back_then_solve(
    "The Freitas paper uses an RW1 random effect for the temporal component. "
    "Why RW1 and not RW2 or AR1? What does that choice imply about the assumed "
    "temporal correlation structure of dengue case counts?"
)

print("STEP 1 - General principle:")
print(result["principle"][:400])

print()

print("STEP 2 - Applied answer:")
print(result["answer"][:400])

**Output (from the article):**

```text
STEP 1 - General principle:
When choosing a temporal random-effect prior for a Bayesian time-series model,
the choice between RW1, RW2, and AR1 reflects different assumptions about how
nearby time points are correlated. RW1 assumes the difference between consecutive
points is mean-zero noise, implying smooth but locally-flexible trajectories.
RW2 assumes the second difference is mean-zero noise - smoother trajectories
that prefer near-linear local trends. AR1 assumes a fixed autocorrelation
parameter, implying a stationary mean-reverting series. The right choice
depends on whether the underlying signal has a drifting mean (RW1, RW2) or
mean-reverting structure (AR1).

STEP 2 - Applied answer:
The Freitas paper picks RW1 because dengue weekly case counts have a clear
drifting mean - peak season vs off-season is not a stationary process - and
because the season-to-season dynamics are flexible enough that the smoother
linear-trend prior of RW2 would oversmooth real epidemic peaks. AR1 would be
wrong because dengue does not mean-revert; outbreak dynamics are persistent.
RW1 lets the model capture both seasonal sweeps and short-term local shocks
without imposing a parametric trend.
```

The step-back call produced a general taxonomy of temporal random-effect priors. The applied call then used that taxonomy to argue specifically why RW1 fits dengue. Without step-back, the model would have given a one-shot answer that was probably right but missed the comparative justification. The pattern forces the model to reason at the right level of abstraction first, which makes the specific answer more defensible.

> This is what Anthropic “plan altitude” guidance is about, make the model reason at the right level of abstraction before it commits to an answer at the specific level.

### Tree of Thoughts and the OODA Subagent Pattern

Techniques #11 and #12. Tree of Thoughts (Yao et al. 2023) generates multiple candidate next-steps at each branch point, scores them, and explores the most promising. The OODA subagent (Boyd’s Observe-Orient-Decide-Act loop, adapted by Anthropic for Claude Code) wraps this into a subagent template that mechanically forces the model through the four phases on every turn.

*ToT (Created by Fareed Khan)*

The OODA pattern is what Claude calls when it dispatches an isolated subagent. The subagent’s system prompt forces it to emit four blocks each iteration: what it observed, how it orients given the contract, what it decides to do, and the actual action.

This is technique #12 verbatim from Anthropic’s published subagent template:

In [ ]:
OODA_SYSTEM_PROMPT = (
    "You are operating in OODA-loop mode. For each turn:\n"
    "  OBSERVE  — what new information arrived from the last tool call?\n"
    "  ORIENT   — given the contract and the DAG, what state are we in?\n"
    "  DECIDE   — what is the single most useful next step?\n"
    "  ACT      — make exactly one tool call (or terminate the loop).\n\n"
    "Output JSON: {\n"
    '  "observation": str,\n'
    '  "orientation": str,\n'
    '  "decision":    str,\n'
    '  "action":      {"tool": str, "args": dict} | {"terminate": str}\n'
    "}\n"
    "Never bundle multiple actions. The loop runs once per OODA turn."
)

We use this with Tree of Thoughts to branch on a hard design question

which inference method should we use for the BYM2+RW1 model, given the constraint that we cannot install R-INLA in our sandbox?

In [ ]:
def tree_of_thoughts(question: str, n_branches: int = 3,
                     depth: int = 1, model: str = MODEL_REASONING) -> dict:
    """Generate n_branches candidate approaches; verifier scores; deepen the best."""
    
    # Generate branches in parallel
    def _gen_branch(i: int) -> str:
        resp = client.chat.completions.create(
            model    = model,
            messages = [
                {"role": "system", "content":
                 "You are exploring a solution branch. Propose ONE specific approach "
                 "with concrete justification. Do not list multiple approaches."},
                {"role": "user", "content": question},
            ],
            temperature = 0.8,
            max_tokens  = 400,
        )
        return resp.choices[0].message.content
    
    with ThreadPoolExecutor(max_workers=n_branches) as ex:
        branches = list(ex.map(_gen_branch, range(n_branches)))
    
    # Score each with the verifier
    scored = []
    for b in branches:
        v = verifier_score(question, b)
        scored.append({"branch": b, **v})
    scored.sort(key=lambda x: x["score"], reverse=True)
    
    return {"winner": scored[0], "all_branches": scored}

Run on the inference-method question:

In [ ]:
result = tree_of_thoughts(
    "We need to fit the BYM2+RW1 hierarchical negative-binomial model from the "
    "Freitas 2025 dengue paper, but R-INLA (the paper's choice) is not "
    "available in our Docker sandbox. What inference method should we use, "
    "and what is the expected accuracy tradeoff?",
    n_branches = 3,
)

print(f"WINNER (score {result['winner']['score']}/10):")
print(result['winner']['branch'][:500])

print()

print("Other branches considered:")

for b in result['all_branches'][1:]:
    print(f"  {b['score']}/10 - {b['branch'][:120]}...")

**Output (from the article):**

```text
WINNER (score 9/10):

Use the Laplace approximation via scipy.optimize.minimize on the
log-posterior, followed by a multivariate-normal posterior sample around the
mode using the inverse Hessian as covariance. This gives the right posterior
mode and approximately right covariance for the BYM2+RW1 spec, which is
sufficient for computing the national 75th-percentile estimate. Expected
deviation from R-INLA's integrated-nested-Laplace approach: 5-10% on tail
quantiles, because INLA's nested integration over hyperparameters is more
accurate than simple Laplace at the mode. Plain scipy is in the standard
container; no extra installs needed.

Other branches considered:
  7/10 - Use PyMC with the NUTS sampler. More accurate than Laplace, closer to R-INLA quality, but installs cleanly via pip. Tradeoff: ~3 minutes per fit vs ~30 seconds for Laplace, and requires...
  4/10 - Bundle R-INLA into the sandbox by installing R via apt and the INLA R package via install.packages('INLA'). Closest to the paper's exact method but takes 10+ minutes to install...
```

Three branches, each a defensible engineering choice. The verifier ranked them. The Laplace branch won at 9/10 because it correctly identified the speed-accuracy tradeoff and named the specific deviation expected (5–10%). The PyMC branch was 7/10 also defensible, but slower and not the optimal first-pass choice. The R-INLA branch was 4/10 the most accurate but operationally expensive given our sandbox constraints.

This is the reasoning that decided the agent’s actual implementation. Phase 8 will show the agent picking Laplace and producing a 7.30% deviation exactly within the predicted 5–10% band. The branch that won the score in Phase 2 is the one that survived contact with reality in Phase 8.

### The Single-Threaded Master Loop with Orchestrator-Worker Topology

Techniques #13, #14, and #15. The master loop is the architectural primitive that everything else hangs off. It is single-threaded by design, one loop, one model, one tool dispatch per turn because every concurrency mechanism (parallel tool calls, subagent spawning) builds on top of the loop without changing its essential shape.

*Master Loop (Created by Fareed Khan)*

Anthropic’s Claude uses one master loop. Period. The orchestrator-worker pattern (technique #15) describes how a parent loop spawns child loops for delegated work, but each child is itself a fresh instance of the same master loop there is no separate “worker loop” with different control flow. This is what makes the architecture composable.

In [ ]:
def master_loop(messages: list, tools: list, dispatch: dict,
                 system: str = STRONG_SYSTEM_PROMPT,
                 model: str = MODEL_FAST,
                 max_iterations: int = 20) -> list:
    """The single-threaded master loop. Pull message → dispatch tools → repeat."""
    
    for iteration in range(max_iterations):
        # Send the current conversation + tools to the model
        resp = client.chat.completions.create(
            model       = model,
            messages    = [{"role": "system", "content": system}] + messages,
            tools       = tools,
            tool_choice = "auto",
            max_tokens  = 2000,
        )
        msg = resp.choices[0].message
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": msg.tool_calls})
        
        # Termination: model produced a final answer with no tool calls
        if not msg.tool_calls:
            print(f"  [loop] terminated at iteration {iteration+1}")
            return messages
        
        # Dispatch every tool call (sequentially in this version; parallel in Phase 5)
        for tc in msg.tool_calls:
            handler = dispatch.get(tc.function.name)
            args    = json.loads(tc.function.arguments)
            try:
                result = handler(**args) if handler else f"Unknown tool: {tc.function.name}"
            except Exception as e:
                result = f"Error: {e}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    
    print(f"  [loop] max_iterations ({max_iterations}) reached")
    return messages

The dispatch map is the only thing that changes between agents. Add a tool, register a handler, the loop never changes:

In [ ]:
BASE_DISPATCH = {
    "search_paper":  lambda query: f"...content from paper.txt matching '{query}'...",
    "list_dataset_columns": lambda: list(cases.columns),
}

BASE_TOOLS = [
    {"type": "function", "function": {
        "name": "search_paper",
        "description": "Search the loaded paper text for a regex query. Returns matched passage with context.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    }},
    {"type": "function", "function": {
        "name": "list_dataset_columns",
        "description": "Return the list of column names in the loaded cases dataframe.",
        "parameters": {"type": "object", "properties": {}},
    }},
]

# Run the loop
result = master_loop(
    messages = [{"role": "user", "content":
                 "What columns are in the dengue dataset, and what does the paper "
                 "say about the casos_prov column specifically?"}],
    tools    = BASE_TOOLS,
    dispatch = BASE_DISPATCH,
)

print(result[-1]["content"][:300])

**Output (from the article):**

```text
[loop] terminated at iteration 3

The dataset has 5 columns: data_iniSE (week start date), municipio_geocodigo
(7-digit municipality code), ID_MN_RESI (residence municipality ID), casos
(confirmed cases), and casos_prov (probable cases - the modelling target).
The paper uses casos_prov throughout because confirmation lags substantially
in the surveillance system; probable cases include both confirmed and
suspected cases that meet the WHO clinical criteria, giving a more
real-time signal of dengue activity.
```

Three iterations: the model called list_dataset_columns, then search_paper, then synthesized the answer. The loop has no special logic about which tool to call when that decision lives entirely in the model. The harness only knows how to dispatch.

This is what Anthropic means by “the loop is the agent”. Every additional capability we add in later phases (parallel tool execution, subagent spawning, persistent state, observability) builds on top of this loop without changing it.

### Sub-Agent Output Discipline

Technique #16. The most important architectural rule for any subagent system: the subagent’s intermediate work never crosses back into the parent’s context. Only the final summary returns. This is what makes Claude able to explore an unfamiliar 50,000-line codebase without the parent conversation drowning in cat and grep output.

*Sub-agent Discipline (Created by Fareed Khan)*

In [ ]:
def spawn_subagent(prompt: str, parent_tools: list, parent_dispatch: dict) -> str:
    """Run a fresh master loop with isolated context. Returns ONLY the final summary."""
    
    SUBAGENT_SYSTEM = (
        "You are a subagent working on a specific subtask. "
        "Complete your task thoroughly. Your output will be the ONLY thing "
        "the parent agent sees — your intermediate tool calls are discarded. "
        "Therefore your final response must be a complete, self-contained summary."
    )
    
    sub_messages = [{"role": "user", "content": prompt}]
    sub_messages = master_loop(
        messages = sub_messages,
        tools    = parent_tools,
        dispatch = parent_dispatch,
        system   = SUBAGENT_SYSTEM,
    )
    
    # Extract ONLY the final assistant message — discard everything else
    for msg in reversed(sub_messages):
        if msg.get("role") == "assistant" and msg.get("content"):
            return msg["content"]
    return "(subagent produced no output)"

The subagent runs the same master_loop as the parent, with the same tools and dispatch, but its messages list starts fresh. Whatever it accumulates file reads, grep outputs, intermediate reasoning stays in sub_messages and gets garbage-collected when the function returns. Only the final string crosses back into the parent.

We test this with a real subtask …

In [ ]:
parent_messages = [
    {"role": "user", "content":
     "What kind of temporal patterns does the dengue dataset show year-over-year? "
     "Look at the data and summarise."}
]

# Parent decides to delegate this exploration to a subagent
exploration_summary = spawn_subagent(
    prompt = ("Examine the cases dataframe (columns: data_iniSE, municipio_geocodigo, "
              "casos_prov). Compute year-over-year totals and identify the seasonal "
              "pattern. Return a 3-sentence summary. Use list_dataset_columns and "
              "any analysis tool you need."),
    parent_tools    = BASE_TOOLS,
    parent_dispatch = BASE_DISPATCH,
)

print("Subagent's summary (the only thing the parent sees):")
print(exploration_summary)

**Output (from the article):**

```text
Subagent's summary (the only thing the parent sees):
Brazilian dengue cases show a strong annual seasonal pattern peaking between
weeks 12-20 (March-May) corresponding to the warm wet season after rainfall
peaks. Year-over-year total cases are highly variable (550K to 2.1M between
2010-2023), with major epidemic years (2013, 2015, 2019, 2022-23) roughly
every 3-5 years driven by serotype dominance shifts. The 2022-2023 season
that the paper forecasts had 1.4M total cases — well above the median but
below the 2015 peak.
```

Three sentences, summarising data exploration that may have involved many internal tool calls inside the subagent. The parent now has a clean, summarised observation in its context not 487,239 rows of dataframe output, not multiple groupby aggregations, just the answer.

This is the pattern that makes long agentic runs possible.

- Without subagent output discipline, every exploration would balloon the parent’s context.

- With it, exploration costs become bounded the parent only pays for the summary, not for the work.

## Phase 3 — Tool-Grounded Execution: From Plan to Verifiable Action

Phase 3 is what turns reasoning and decomposition into verifiable action the model produces concrete plans, executes them through tools, and corrects itself when reality contradicts its expectations.

*Phase (Tool Execution) Created by Fareed Khan*

Claude published architecture follows this pattern explicitly. Plan-and-execute (Wang et al. 2023) separates planning from execution. ReAct (Yao et al. 2022) interleaves them. Reflexion (Shinn et al. 2023) adds self-correction. We will build all three.

### Plan-and-Execute and the LLM Compiler

Techniques #17 and #18. Plan-and-execute is the simplest case: the model produces a complete plan upfront, then executes each step. The LLM Compiler (Kim et al. 2023) extends this by identifying which steps in the plan are independent and dispatching them in parallel.

*Plan and execute (Created by Fareed Khan)*

For paper reproduction, plan-and-execute matters because the steps have a strict dependency order you cannot fit the model before you have built the spatial adjacency, you cannot build the adjacency before you have aggregated the data. Getting the plan right upfront avoids backtracking later:

In [ ]:
import json
from pydantic import BaseModel
from typing import List

class PlanStep(BaseModel):
    step_id: str
    description: str
    depends_on: List[str]
    expected_artifact: str

class Plan(BaseModel):
    goal: str
    steps: List[PlanStep]

def make_plan(goal: str, model: str = MODEL_REASONING) -> Plan:
    """Have the model produce a structured, dependency-ordered plan."""
    PLAN_SYSTEM = (
        "Produce a step-by-step plan to achieve the goal. Each step must have:\n"
        "  - A short step_id like 's1', 's2', etc.\n"
        "  - A description of what the step does\n"
        "  - depends_on: list of step_ids that must complete first\n"
        "  - expected_artifact: what file/value the step produces\n"
        "Output JSON matching this schema:\n"
        '{"goal": str, "steps": [{"step_id":..., "description":..., '
        '"depends_on": [...], "expected_artifact":...}]}'
    )
    resp = client.chat.completions.create(
        model    = model,
        messages = [
            {"role": "system", "content": PLAN_SYSTEM},
            {"role": "user",   "content": goal},
        ],
        response_format = {"type": "json_object"},
        temperature = 0.0,
        max_tokens  = 1500,
    )
    return Plan.model_validate_json(resp.choices[0].message.content)

Run it on the real reproduction goal:

In [ ]:
plan = make_plan(
    "Reproduce the Freitas et al. 2025 dengue paper. Fit BYM2+RW1 hierarchical "
    "negative-binomial model on 12 training seasons (2010-2022), forecast "
    "2022-2023, validate against paper's reported 1,405,191 national p75."
)

print(f"GOAL: {plan.goal}\n")

for s in plan.steps:
    deps = ", ".join(s.depends_on) or "(root)"
    print(f"  [{s.step_id}] {s.description}")
    print(f"        depends_on: {deps}")
    print(f"        produces:   {s.expected_artifact}\n")

**Output (from the article):**

```text
GOAL: Reproduce Freitas 2025 dengue paper end-to-end with verified national p75.

  [s1] Load DATASUS dengue cases CSV and filter to 2010-2024 weekly totals
        depends_on: (root)
        produces:   load_data.py + cases_filtered DataFrame

  [s2] Aggregate municipality-level cases to health-district x epi-week
        depends_on: s1
        produces:   aggregate.py + district_week DataFrame (118 districts × 52 weeks)

  [s3] Build the 118x118 binary adjacency matrix from district shapefiles
        depends_on: s2
        produces:   adjacency.py + W_adj numpy array

  [s4] Specify the BYM2 + RW1 hierarchical negative-binomial model
        depends_on: s3
        produces:   model.py with build_model() returning model spec dict

  [s5] Implement and run inference (Laplace approximation via scipy)
        depends_on: s4
        produces:   inference.py + posterior samples on disk

  [s6] Compute national 75th-percentile from per-district posterior samples
        depends_on: s5
        produces:   validate.py + national_p75 number

  [s7] Run spec-layer validation against paper Table 2 target (1,405,191 ± 5%)
        depends_on: s6
        produces:   SPEC_LAYER_REPORT.json

  [s8] Generate REPORT.md with verdict (reproduces / partial / fails)
        depends_on: s7
        produces:   REPORT.md
```

The plan is structurally identical to what an experienced researcher would write down on a napkin before starting. The dependency edges are non-trivial — s4 needs s3's adjacency matrix, s7 needs s6's computed number. The agent in Phase 8 will execute exactly these 8 subgoals in this order.

The LLM Compiler extension would identify that s2 and s3 could technically run in parallel (both depend only on s1, but on different aspects of it). For our paper this turns out to not matter they take seconds but for a paper with 50 independent feature-engineering steps this would compress wall time dramatically.

### ReAct, Evaluator-Optimizer, and Reflexion: The Self-Correction Family

Techniques #19, #20, and #21. These three patterns all address the same problem:

> what does the agent do when its first attempt is wrong?

*ReAct Loop (Created by Fareed Khan)*

ReAct (Reason + Act, Yao et al. 2022) interleaves reasoning steps with tool calls. The model thinks, then acts, then sees the result of the action, then thinks again. It is the canonical agent loop pattern. We have effectively been building ReAct since Phase 2 the master loop is a ReAct loop.

- Evaluator-Optimizer (Anthropic, Building Effective Agents, December 2024) wraps the loop with an explicit evaluator that decides whether the current output is good enough to commit. If not, the loop iterates with the evaluator’s critique as feedback.

- Reflexion (Shinn et al. 2023) adds episodic memory of past failures. After each failed attempt, the model writes a verbal lesson “last time I tried X and it produced Y, the issue was Z” into a memory store. On the next attempt, it reads its own previous lessons and incorporates them.

We build the evaluator-optimizer loop here:

In [ ]:
EVALUATOR_SYSTEM = (
    "You are evaluating whether the candidate output meets the goal. "
    "Be strict. If the output is missing a required component, say so specifically. "
    "Output JSON: {\"accept\": bool, \"critique\": str (specific issues if not accepted)}."
)

def evaluator_optimizer(goal: str, generator_fn, max_rounds: int = 3) -> dict:
    """Iterate generator_fn → evaluator until accepted or max_rounds hit."""
    history = []
    feedback = None

    for round_num in range(1, max_rounds + 1):

        # Generate (with feedback from previous round if any)
        prompt = goal if not feedback else f"{goal}\n\nPREVIOUS ATTEMPT'S ISSUES:\n{feedback}"
        candidate = generator_fn(prompt)
        
        # Evaluate
        eval_resp = client.chat.completions.create(
            model    = MODEL_REASONING,
            messages = [
                {"role": "system", "content": EVALUATOR_SYSTEM},
                {"role": "user",   "content": f"GOAL:\n{goal}\n\nCANDIDATE:\n{candidate}"},
            ],
            response_format = {"type": "json_object"},
            temperature = 0.0,
            max_tokens  = 300,
        )

        verdict = json.loads(eval_resp.choices[0].message.content)
        
        history.append({"round": round_num, "candidate": candidate, **verdict})
        
        if verdict["accept"]:
            return {"final": candidate, "rounds_used": round_num,
                    "status": "accepted", "history": history}
        feedback = verdict["critique"]
    
    return {"final": history[-1]["candidate"], "rounds_used": max_rounds,
            "status": "max_rounds_hit", "history": history}

We test this on a generation task that typically takes multiple iterations to land:

In [ ]:
def generator(prompt):
    return think_then_answer(prompt, max_tokens=500).answer

result = evaluator_optimizer(
    goal = ("Write a 4-sentence summary of the BYM2 spatial random-effect prior "
            "for a methods section. It MUST include: the Riebler 2016 citation, "
            "the role of the mixing parameter phi, the PC prior choice "
            "(U=1, alpha=0.01), and contrast with the plain BYM model."),
    generator_fn = generator,
    max_rounds = 3,
)

print(f"Status: {result['status']}, rounds used: {result['rounds_used']}\n")

for h in result["history"]:
    accepted = "ACCEPTED" if h["accept"] else "REJECTED"

    print(f"--- Round {h['round']} [{accepted}] ---")

    if not h["accept"]:
        print(f"  critique: {h['critique']}")
    print(f"  candidate: {h['candidate'][:200]}...")
    print()

**Output (from the article):**

```text
Status: accepted, rounds used: 2

--- Round 1 [REJECTED] ---
  critique: Missing the explicit PC prior choice (U=1, alpha=0.01) and does not contrast against the plain BYM model. Mentions Riebler but does not cite the 2016 year.
  candidate: The BYM2 spatial random-effect prior decomposes spatial variation into a structured CAR component and an unstructured iid component...
--- Round 2 [ACCEPTED] ---
  candidate: Following Riebler et al. (2016), we use the BYM2 prior which decomposes the spatial random effect into a structured CAR and an unstructured iid component, weighted by a mixing parameter phi in [0,1]. Penalised-complexity priors with U=1 and alpha=0.01 are placed on both the precision and phi, encoding a weak preference toward the simpler iid model. Unlike the plain BYM specification of Besag et al., BYM2's parameterisation rescales the precision so the prior is invariant to the graph topology, making it directly comparable across different spatial structures.
```

The first attempt missed two specific requirements. The evaluator named them explicitly. The second attempt addressed them. Without the evaluator, the first round’s answer would have shipped it looked plausible but it was missing the citation year and the contrast that the methods section actually needs.

This is the inner loop of how Claude produces methods sections, READMEs, and any prose-heavy artifact. The evaluator catches what looks like good output but is missing required elements. Iteration cost is bounded; quality ceiling is much higher.

### CRITIC and Mixture-of-Agents

Techniques #22 and #23. CRITIC (Gou et al. 2023) grounds critique in real tool outputs rather than just LLM judgment the critic actually runs the candidate code, queries the actual database, fetches the actual webpage. Mixture-of-Agents (Wang et al. 2024) uses an ensemble of generators with different framings to produce diverse candidates, then aggregates with a final synthesizer.

*MoA (Created by Fareed Khan)*

For Brevity here we will skip CRITIC’s full implementation Phase 4 external-feedback verification covers the same architectural pattern with real pytest execution. Mixture-of-Agents is interesting because it produces qualitatively different candidates rather than just samples around the same answer:

In [ ]:
def mixture_of_agents(query: str, framings: list[str]) -> dict:
    """Generate candidates with different role framings, synthesize a final answer."""
    
    # Stage 1: generate K candidates with different role framings
    def _gen_with_framing(framing):
        framed = f"Adopt this perspective: {framing}\n\nQUESTION:\n{query}"
        return think_then_answer(framed, model=MODEL_FAST, temperature=0.4).answer
    
    with ThreadPoolExecutor(max_workers=len(framings)) as ex:
        candidates = list(ex.map(_gen_with_framing, framings))
    
    # Stage 2: synthesize using the strong model
    synthesizer_prompt = (
        "Multiple expert perspectives have produced candidate answers. Synthesize "
        "the strongest combined answer that incorporates the best insights from each.\n\n"
        + "\n\n".join(f"PERSPECTIVE {i+1} ({framings[i]}):\n{c}"
                       for i, c in enumerate(candidates))
    )
    synth_resp = client.chat.completions.create(
        model    = MODEL_REASONING,
        messages = [{"role": "user", "content": synthesizer_prompt}],
        max_tokens = 800,
    )
    
    return {"candidates": candidates, "synthesis": synth_resp.choices[0].message.content}

We use it on a design question with multiple defensible framings:

In [ ]:
result = mixture_of_agents(
    query = ("How should we structure the 8 subgoals of the Freitas reproduction "
             "into a directory layout? What goes in agent_code/, what goes in "
             "data/, what goes in reports/?"),
    framings = [
        "a Python package maintainer optimizing for clean imports",
        "a research engineer who needs to rerun this on different papers",
        "a reproducibility reviewer auditing the codebase",
    ],
)

print("SYNTHESIZED ANSWER:")
print(result["synthesis"][:600])

**Output (from the article):**

```text
SYNTHESIZED ANSWER:
The cleanest structure separates code, data, and outputs at the top level
while keeping each agent-written module importable as a flat module:


agent_code/         - every .py file the agent writes, flat namespace
    load_data.py
    aggregate.py
    adjacency.py
    model.py
    inference.py
    validate.py
  data/               - read-only inputs (cases.csv.gz, spatial.tbl.csv)
  reports/            - outputs the agent generates
    figure_observed_vs_forecast.png
    REPORT.md
  agent_code/SPEC_LAYER_REPORT.json     - verdict, lives next to code
  agent_code/DEFINITION_OF_DONE.json    - contract, also next to code

The package-maintainer view favours flat agent_code/ for clean imports.
The research-engineer view favours separating data/ from agent_code/ so the
same agent code runs on a different dataset. The reviewer view favours
having reports/ separate so reviewers see outputs without scrolling code.
This layout satisfies all three.
```

The synthesizer combined the three perspectives into a single layout that all three would endorse and crucially, the synthesizer named which constraint each part of the layout satisfies. This is the layout we are using throughout the paper, exactly as produced. The mixture pattern produced a strictly better design than any single perspective would have on its own.

### Verifier Asymmetry

Technique #24. The single most important architectural insight from Phase 1’s best-of-N work, formalized: generating is harder than verifying. A 7B model can reliably check whether code is correct even when it cannot reliably write correct code. This means we can use a cheap generator and a strong verifier, paying for verification only on the final candidates.

*Verifier Asymmetry (Created by Fareed Khan)*

This is the principle behind Claude design: Sonnet generates, the test runner verifies. The verifier is not another LLM on critical paths it is pytest. We will see this in Phase 7's spec layer.

But for design questions where pytest does not apply, the asymmetric LLM verifier still works:

In [ ]:
def asymmetric_solve(query: str, n_candidates: int = 4) -> dict:
    """Cheap generator (n candidates) + strong verifier (single ranking call)."""
    
    # Cheap generator at temperature 0.7 for diversity
    with ThreadPoolExecutor(max_workers=n_candidates) as ex:
        candidates = list(ex.map(
            lambda _: think_then_answer(query, model=MODEL_FAST, temperature=0.7).answer,
            range(n_candidates),
        ))
    
    # Single strong-model call ranks all of them
    rank_prompt = (
        "Rank these candidates from best to worst. Output JSON: "
        "{\"ranking\": [{\"rank\": 1, \"index\": 0, \"reason\": ...}, ...]}\n\n"
        + "\n\n".join(f"CANDIDATE {i}:\n{c}" for i, c in enumerate(candidates))
    )
    rank_resp = client.chat.completions.create(
        model    = MODEL_REASONING,
        messages = [{"role": "user", "content": rank_prompt}],
        response_format = {"type": "json_object"},
        max_tokens = 400,
    )
    ranking = json.loads(rank_resp.choices[0].message.content)["ranking"]
    
    winner_idx = ranking[0]["index"]
    return {
        "winner":  candidates[winner_idx],
        "winner_reason": ranking[0]["reason"],
        "ranking": ranking,
    }

The cost arithmetic on this matters. Generating 4 candidates at the cheap rate costs ~4× one cheap call. Verifying with one strong-model call costs ~5× a cheap call. Total: ~9× a single cheap call. Compare to running the strong model 4 times: ~20× a cheap call. Asymmetric solve gives us best-of-4 strong-model quality at ~half the cost.

In [ ]:
result = asymmetric_solve(
    "Design the function signature for the agent's `aggregate_to_district_week()` "
    "function. It takes the cases dataframe and a spatial lookup dataframe and "
    "produces a district-week aggregation. Show the function signature with type "
    "hints and a docstring.",
    n_candidates = 4,
)

print(f"WINNER (chosen because: {result['winner_reason'][:100]}...):")
print(result["winner"][:400])

**Output (from the article):**

```text
WINNER (chosen because: Uses pandas type hints from typing.TYPE_CHECKING for clean imports, has a complete docstring with column expectations...):

import pandas as pd

from typing import TYPE_CHECKING
def aggregate_to_district_week(
    cases_df: 'pd.DataFrame',
    spatial_lookup_df: 'pd.DataFrame',
) -> 'pd.DataFrame':
    """Aggregate municipality-level cases to district x epi-week granularity.
    Parameters
    ----------
    cases_df : DataFrame with columns [data_iniSE, municipio_geocodigo, casos_prov]
    spatial_lookup_df : DataFrame with columns [municipio_geocodigo, regional_saude_id]
    Returns
    -------
    DataFrame with columns [regional_saude_id, epi_week, casos_prov]
    where epi_week is the renamed data_iniSE column.
    """
    ...
```

The verifier picked the candidate with the cleanest type-hint pattern, the most complete docstring, and the most unambiguous column-expectation documentation. The other three were not bad; the verifier picked the best. This is the function signature the agent will actually write in Phase 8.

## Phase 4 — Production Reliability: The Hardening Stack

The first three phases gave us a thinking, decomposing, self-correcting agent. None of that matters in production if the agent loses 5% of its runs to silent failures.

> Going from 80% reliable to 99% reliable is not 25% more work, it is 5× more work, because the last 19% of failures are all different corner cases that each need their own pattern.

*Phase 4 (Production Reliability) Created by Fareed Khan*

This is the layer where the harness earns its production claim. Claude itself, as a frontier reasoning system, ships with these patterns enabled by default. When you ask Claude to write a 500 line analysis, it does not produce 500 lines and stop. It internally drafts, critiques, refines, and only then emits the final output. The user sees a polished result, the harness around the model has done four passes of editorial discipline. We are going to make those passes explicit at the inference layer.

### Self Refine, Verifier Guided Search, and External Feedback Verification

Techniques #25, #26, and #27. These three patterns form a ladder of correctness checking, from cheapest to strongest.

*Self-Refine (Created by Fareed Khan)*

- Self Refine (Madaan et al. 2023) uses the same model to critique and refine its own output for K iterations. It is the cheapest option, because no separate verifier is involved. It works for polish heavy tasks where the answer just needs to get sharper.

- Verifier Guided Search (Snell et al. 2024) generates K candidates, scores them with a separate verifier, and rejects the round entirely if no candidate clears the threshold, generating a fresh batch. This is technique #26 and it is what stops the agent from committing to a sub par answer just because best of N picked the least bad option.

External Feedback Verification is the strongest. Instead of asking another LLM whether the candidate is correct, the harness runs the code and uses real test execution as the verdict. This is technique #27 and it is the gold standard. Claude internally uses this whenever a tool result can verify the model’s claim mechanically.

Self Refine first, because it is the simplest:

In [ ]:
SELF_CRITIQUE_PROMPT = (
    "You wrote the following output. Now critique it as if you were a "
    "strict reviewer. Identify SPECIFIC issues: missing pieces, unclear "
    "sections, errors, awkward phrasing. Be concise, list 2-5 specific "
    "issues. If the output is already excellent, say so."
)

SELF_REFINE_PROMPT = (
    "Here is your previous output:\n{previous}\n\n"
    "Here is your own critique of it:\n{critique}\n\n"
    "Now produce a refined version that addresses every point in the critique."
)

def self_refine(initial_query: str, iterations: int = 3,
                 model: str = MODEL_FAST) -> dict:
    """K iterations of (same model: generate then critique then refine)."""
    current = think_then_answer(initial_query, model=model, max_tokens=600).answer
    history = [{"iteration": 0, "output": current, "critique": None}]
    
    for k in range(1, iterations + 1):
        critique_resp = client.chat.completions.create(
            model    = model,
            messages = [
                {"role": "user",      "content": initial_query},
                {"role": "assistant", "content": current},
                {"role": "user",      "content": SELF_CRITIQUE_PROMPT},
            ],
            temperature = 0.3,
            max_tokens  = 400,
        )
        critique = critique_resp.choices[0].message.content
        
        refine_resp = client.chat.completions.create(
            model    = model,
            messages = [
                {"role": "user", "content": initial_query},
                {"role": "user", "content": SELF_REFINE_PROMPT.format(previous=current, critique=critique)},
            ],
            temperature = 0.3,
            max_tokens  = 600,
        )
        current = refine_resp.choices[0].message.content
        history.append({"iteration": k, "output": current, "critique": critique})
    
    return {"final": current, "history": history, "iterations_run": iterations}

We run this on a real task from our reproduction pipeline, writing a brief README for the agent_code directory:

In [ ]:
result = self_refine(
    "Write a brief README.md for the agent_code directory of our Brazilian "
    "dengue reproduction project. It should explain what the directory contains, "
    "how to run the code, and how to verify the reproduction succeeded. "
    "Keep it under 200 words.",
    iterations = 3,
)

print("Length progression across iterations:")

for h in result["history"]:
    print(f"  iter {h['iteration']}: {len(h['output'])} chars")

print("\nCritique themes per iteration:")
for h in result["history"]:
    if h["critique"]:
        first_line = h["critique"].split("\n")[0]
        print(f"  iter {h['iteration']}: {first_line[:100]}")

**Output (from the article):**

```text
Length progression across iterations:
  iter 0: 387 chars
  iter 1: 612 chars
  iter 2: 689 chars
  iter 3: 698 chars

Critique themes per iteration:
  iter 1: Missing dependency list. The 'how to verify' section is too vague, should mention the 1,436,034
  iter 2: Verification step now mentions 1,436,034 but should also state the tolerance and the source of truth
  iter 3: Excellent, covers all required sections, has clear verification criteria, includes troubleshooting
```

Three iterations, each one strictly improving on the previous. The length grew from 387 to 698 chars, but more importantly each critique addressed something specific that was missing. By iteration 3 the critique returned “excellent”, which is the convergence signal. Further iterations would just refine around the noise floor and waste budget.

The convergence detection is what makes self refine production grade. Without a stopping condition, the loop runs forever; with one, it stops at the right moment and saves API spend.

External Feedback Verification is the strongest pattern because it removes the LLM from the verifier entirely. We have the agent write code, then we run that code in a real Python REPL, and the test result is the verdict:

In [ ]:
import io
import contextlib
import traceback


class PythonREPL:
    """Stateful Python REPL. State persists across run() calls."""
    def __init__(self, preloaded: dict = None):
        self.namespace = preloaded or {}
    
    def run(self, code: str) -> dict:
        stdout = io.StringIO()
        try:
            with contextlib.redirect_stdout(stdout):
                exec(code, self.namespace)
            return {"stdout": stdout.getvalue(), "error": None,
                     "value": self.namespace.get("_result")}
        except Exception:
            return {"stdout": stdout.getvalue(),
                     "error": traceback.format_exc(), "value": None}

def external_feedback_verify(candidate_code: str, test_code: str) -> dict:
    """Run candidate code, then run test code. Both share a fresh REPL namespace."""
    repl = PythonREPL(preloaded={"cases": cases, "pd": pd})
    
    cand_result = repl.run(candidate_code)
    if cand_result["error"]:
        return {"passed": False, "phase": "candidate_code",
                 "error": cand_result["error"]}
    
    test_result = repl.run(test_code)
    if test_result["error"]:
        return {"passed": False, "phase": "test_code",
                 "error": test_result["error"]}
    
    return {"passed": True, "phase": "all", "output": test_result["stdout"]}

def code_with_tests(code_gen_question: str, test_code: str,
                     max_rounds: int = 3) -> dict:
    """Generate code, run real tests, regenerate with failure feedback if tests fail."""
    feedback = ""
    history = []
    for round_num in range(1, max_rounds + 1):
        prompt = code_gen_question + (f"\n\nPREVIOUS ATTEMPT FAILED:\n{feedback}" if feedback else "")
        prompt += "\n\nOutput ONLY raw Python code. No markdown fences."
        candidate = think_then_answer(prompt, max_tokens=800).answer
        
        if "```" in candidate:
            inner = candidate.split("```")[1]
            if inner.startswith("python"):
                inner = inner[6:]
            candidate = inner.strip()
        
        verify_result = external_feedback_verify(candidate, test_code)
        history.append({"round": round_num, "code": candidate, "verify": verify_result})
        
        if verify_result["passed"]:
            return {"final_code": candidate, "rounds_used": round_num,
                     "status": "passed", "history": history}
        feedback = f"phase={verify_result['phase']}, error={verify_result['error']}"
    
    return {"final_code": history[-1]["code"], "rounds_used": max_rounds,
             "status": "failed_after_max_rounds", "history": history}

We run this on a real generation task that has a known correct answer (the 2022 to 2023 season total of 1,436,034):

In [ ]:
code_question = (
    "Write a Python function `season_total(cases, start, end)` that takes the "
    "cases dataframe plus two ISO date strings, filters to where data_iniSE is "
    "between start and end inclusive, and returns the sum of the casos_prov column as int."
)

test_assertion = (
    "actual = season_total(cases, '2022-10-09', '2023-10-01')\n"
    "expected = 1_436_034\n"
    "assert actual == expected, f'expected {expected}, got {actual}'\n"
    "print('PASS')"
)

result = code_with_tests(code_question, test_assertion, max_rounds=3)
print(f"Status: {result['status']}, rounds used: {result['rounds_used']}\n")

for h in result["history"]:
    v = h["verify"]
    print(f"  Round {h['round']}: passed={v['passed']}, phase={v['phase']}")
    if not v["passed"]:
        print(f"    error: {v['error'][:120]}")
    else:
        print(f"    test stdout: {v['output'].strip()}")

**Output (from the article):**

```text
Status: passed, rounds used: 2

Round 1: passed=False, phase=test_code
    error: AssertionError: expected 1436034, got 1437291
  Round 2: passed=True, phase=all
    test stdout: PASS
```

Round 1 produced syntactically valid Python that compiled and ran without error. An LLM verifier looking at that code would have probably scored it 7 or 8 out of 10, it looks like correct pandas. The agent might have committed to it.

The test caught it. Off by 1257 rows somewhere, probably a strict less than instead of less than or equal in the date filter. No LLM verifier reading the code would have noticed; only running the code did.

Round 2, given the real failure message as feedback, fixed it. The test now prints PASS.

This is exactly how Claude’s architecture handles code generation internally. When Claude writes a Python function and a test exists for it, the harness around Claude runs the test. If the test fails, the failure message goes back into Claude’s context and it tries again. Anthropic published SWE-Bench-Verified results, which show Claude going from around 38% on single shot to 80.8% with this loop, are entirely produced by this pattern.

### Tool Description Self Improvement and Adversarial Self Probing

Techniques #28 and #29. These two are the patterns that catch what external tests miss.

Anthropic’s June 2025 multi agent research blog revealed a counterintuitive finding from their Claude Code work.

> The largest single quality jump they got from prompt engineering was not from improving system prompts. It was from improving tool descriptions.

*Tools (Created by Fareed Khan)*

When Claude misused a tool, the issue was almost always that the tool’s description was ambiguous, leaving Claude to guess. Asking the model itself “what is wrong with this tool’s description given these observed misuses” produced descriptions that reduced misuse by 40 percent or more.

Adversarial Self Probing is the other side. Instead of asking the model “is this output good”, we ask “find ways to break this output”. Adversarial framing produces qualitatively different critiques than constructive framing. Constructive critiques find style issues. Adversarial critiques find bugs.

In [ ]:
ADVERSARIAL_SYSTEM = (
    "You are a hostile adversary. Your job is to find ways to BREAK "
    "the candidate output. Look for: (1) edge cases that produce wrong "
    "results, (2) implicit assumptions that may not hold, (3) concrete "
    "counterexamples, (4) failure modes that are not handled. "
    "Be specific. Each attack must include the exact input or scenario "
    "that would trigger it. "
    "Output JSON: {\"attacks\": [{\"category\": str, \"scenario\": str, "
    "\"why_it_breaks\": str, \"severity\": \"critical\"|\"major\"|\"minor\"}]}."
)

def adversarial_probe(target_description: str, candidate_output: str,
                       n_attacks_max: int = 5) -> list[dict]:
    user = (
        f"TARGET (what the output should accomplish):\n{target_description}\n\n"
        f"CANDIDATE OUTPUT TO ATTACK:\n{candidate_output}\n\n"
        f"Find up to {n_attacks_max} ways to break this output."
    )
    resp = client.chat.completions.create(
        model = MODEL_REASONING,
        messages = [
            {"role": "system", "content": ADVERSARIAL_SYSTEM},
            {"role": "user",   "content": user},
        ],
        response_format = {"type": "json_object"},
        temperature = 0.4,
        max_tokens  = 800,
    )
    return json.loads(resp.choices[0].message.content)["attacks"]

We run this on the same season_total function from above that already passed the external feedback test:

In [ ]:
candidate_code = """
def season_total(cases, start, end):
    s = pd.Timestamp(start)
    e = pd.Timestamp(end)
    mask = (cases['data_iniSE'] >= s) & (cases['data_iniSE'] <= e)
    return int(cases.loc[mask, 'casos_prov'].sum())
"""

target_desc = (
    "A function season_total(cases, start, end) that filters dengue cases to a "
    "season window and returns the total. Must work robustly for any valid window."
)

attacks = adversarial_probe(target_desc, candidate_code, n_attacks_max=4)

print(f"Adversarial probe found {len(attacks)} attacks on code that passed pytest:\n")

severity_order = {"critical": 0, "major": 1, "minor": 2}
attacks_sorted = sorted(attacks, key=lambda a: severity_order.get(a["severity"], 3))

for a in attacks_sorted:
    print(f"[{a['severity']}] {a['category']}")
    print(f"  Scenario: {a['scenario']}")
    print(f"  Why it breaks: {a['why_it_breaks']}")
    print()

**Output (from the article):**

```text
Adversarial probe found 4 attacks on code that passed pytest:

[critical] input_validation
  Scenario: Caller passes start='2023-10-01', end='2022-10-09' (reversed dates).
  Why it breaks: Function silently returns 0 with no error. The caller assumes failure means an empty season, not malformed input.

[major] dtype_coercion
  Scenario: Caller passes start as a non-ISO string like 'October 9, 2022'.
  Why it breaks: pd.Timestamp accepts loose formats but the resulting comparison may produce surprising results across pandas versions and locales.

[major] no_validation_when_caller_misuses_default
  Scenario: Caller uses non-default dates like the 2023-2024 season.
  Why it breaks: The pytest assertion only validates the default 2022-2023 window. Non-default calls have no validation, providing false confidence.

[minor] file_not_found_handling
  Scenario: cases dataframe is empty or has zero rows in the window.
  Why it breaks: Returns 0, indistinguishable from a valid window with zero cases.
```

External feedback catches what the test exercises. Adversarial framing finds the gaps the test does not cover.

This is exactly the pattern Claude’s training pipeline includes. There is a “red team” stage where one Claude actively tries to elicit broken behaviour from another Claude. That is this technique applied at training time. We get a meaningful fraction of that benefit at inference time, without any retraining.

### The Editorial Discipline Triad: Architect Editor, Linter, Result Compaction

Techniques #30, #31, and #32. These three together are what Anthropic calls editorial discipline, the patterns that prevent problems instead of detecting them.

*Editorial Discipline (Created by Fareed Khan)*

Architect Editor Split uses a strong model to design and a cheap model to implement. The strong model spends its expensive tokens on structural decisions, the cheap model fills in mechanical work. Aider’s open source agent reports that this split delivers strong model quality at roughly 30 percent of the cost.

Linter in the Loop automatically reverts any commit that fails static checks. Bad code never reaches the visible state of the codebase.

> Tool Result Compaction summarizes long tool outputs before they re enter the conversation, preventing context bloat.

We build the architect editor split first, because it is the highest leverage of the three:

In [ ]:
ARCHITECT_SYSTEM = (
    "You are a senior architect. Given a task, produce a STRUCTURED PLAN "
    "that the editor will implement. Do NOT produce the final output. "
    "Produce the PLAN. "
    "Output JSON: {\"plan\": [{\"section\": str, \"intent\": str, "
    "\"key_constraints\": [str]}], \"design_decisions\": [{\"decision\": "
    "str, \"rationale\": str}]}. "
    "Be specific about what each section must contain. Be ruthless about constraints."
)

EDITOR_SYSTEM = (
    "You are an editor. The architect has produced a structured plan. "
    "Your job is to execute it, produce the actual output that satisfies "
    "the plan. Do NOT redesign. Do NOT add new sections or skip planned "
    "ones. Follow the plan precisely. Output the final result only."
)

def architect_editor_solve(task: str, editor_max_tokens: int = 1500) -> dict:
    """Architect (strong) plans; Editor (cheap) implements."""
    arch_resp = client.chat.completions.create(
        model = MODEL_REASONING,
        messages = [
            {"role": "system", "content": ARCHITECT_SYSTEM},
            {"role": "user",   "content": f"TASK:\n{task}"},
        ],
        response_format = {"type": "json_object"},
        temperature = 0.2,
        max_tokens  = 800,
    )
    plan = json.loads(arch_resp.choices[0].message.content)
    arch_tokens = arch_resp.usage.completion_tokens
    
    plan_str = json.dumps(plan, indent=2)
    edit_resp = client.chat.completions.create(
        model = MODEL_FAST,
        messages = [
            {"role": "system", "content": EDITOR_SYSTEM},
            {"role": "user",   "content": f"TASK:\n{task}\n\nARCHITECT PLAN:\n{plan_str}\n\nProduce the final output now."},
        ],
        temperature = 0.3,
        max_tokens  = editor_max_tokens,
    )
    edit_tokens = edit_resp.usage.completion_tokens
    
    return {
        "plan":   plan,
        "output": edit_resp.choices[0].message.content,
        "architect_tokens": arch_tokens,
        "editor_tokens":    edit_tokens,
    }

We run this on a codegen task from our reproduction:

In [ ]:
task = (
    "Write the contents of agent_code/aggregate.py, a Python module containing a "
    "single function aggregate_to_district_week(cases_df, spatial_lookup_df) that "
    "aggregates the cases dataframe to health-district by epi-week granularity. "
    "Output ONLY the raw Python source code."
)

result = architect_editor_solve(task, editor_max_tokens=600)

print("ARCHITECT (strong model) plan:")
print(json.dumps(result["plan"], indent=2)[:600])
print()
print("EDITOR (cheap model) output:")
print(result["output"][:300])
print()

arch_cost = result["architect_tokens"] * 2.19e-6
edit_cost = result["editor_tokens"]    * 1.10e-6
hyp_strong_only = (result["architect_tokens"] + result["editor_tokens"]) * 2.19e-6
savings = (hyp_strong_only - (arch_cost + edit_cost)) / hyp_strong_only * 100

print(f"Architect tokens: {result['architect_tokens']}")
print(f"Editor tokens:    {result['editor_tokens']}")
print(f"Total cost:       ${arch_cost + edit_cost:.4f}")
print(f"If strong-only:   ${hyp_strong_only:.4f}")
print(f"Saving:           {savings:.0f}%")

**Output (from the article):**

```text
ARCHITECT (strong model) plan:
{
  "plan": [
    {
      "section": "imports",
      "intent": "Pull in pandas; nothing else",
      "key_constraints": ["single import line"]
    },
    {
      "section": "function_signature",
      "intent": "Define aggregate_to_district_week(cases_df, spatial_lookup_df) -> DataFrame",
      "key_constraints": ["both args are DataFrames", "return DataFrame with columns [district_id, epi_week, casos_prov]"]
    },
    {
      "section": "merge_step",
      "intent": "Inner-merge cases with spatial_lookup on the municipality code",
      "key_constraints": ["use municipio_geocodigo as join key", "inner merge to drop municipalities without district mapping"]
    }
  ],
  "design_decisions": [
    {"decision": "inner merge instead of left", "rationale": "municipalities without a district mapping should not appear in district-level analysis"}
  ]
}


EDITOR (cheap model) output:

import pandas as pd
def aggregate_to_district_week(cases_df, spatial_lookup_df):
    merged = cases_df.merge(spatial_lookup_df, on='municipio_geocodigo', how='inner')
    grouped = (merged.groupby(['district_id', 'data_iniSE'])['casos_prov']
                     .sum()
                     .reset_index()
                     .rename(columns={'data_iniSE': 'epi_week'}))
    return grouped

Architect tokens: 287
Editor tokens:    124
Total cost:       $0.0014
If strong-only:   $0.0029
Saving:           52%
```

The architect produced a 5 section plan with explicit design decisions. The editor implemented it mechanically without redesigning anything. The 52% cost saving comes from a structural fact: writing pandas code is cheap in cognitive terms, the cheap model can do it perfectly.

> Designing the function signature and choosing inner versus outer merge is expensive, requires structural reasoning. Splitting the work matches each model to its strength.

This is exactly what Anthropic does internally with Claude Haiku, Sonnet, and Opus. The three tier model pricing only makes economic sense if users build architect editor pipelines. Claude itself, in extended thinking mode, runs an internal architect editor pattern within a single response.

The Linter in the Loop is the next pattern. It is structurally simpler but high impact:

In [ ]:
import py_compile
import ast
import tempfile
import os

def lint_python(code: str) -> dict:
    """Run lightweight static checks. Returns {'passed': bool, 'errors': [str]}."""
    errors = []
    
    # Syntax check via py_compile
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w") as tmp:
        tmp.write(code)
        tmp_path = tmp.name
    try:
        py_compile.compile(tmp_path, doraise=True)
    except py_compile.PyCompileError as e:
        errors.append(f"SyntaxError: {e}")
    finally:
        os.unlink(tmp_path)
    
    # AST based smell checks
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.ExceptHandler) and node.type is None:
                errors.append(f"Style: bare `except:` at line {node.lineno}")
    except SyntaxError:
        pass
    
    return {"passed": len(errors) == 0, "errors": errors}

def safe_write_code_file(filename: str, content: str) -> str:
    """Write file ONLY if it lints clean. Otherwise revert and return error."""
    if not filename.endswith(".py") or "/" in filename or ".." in filename:
        return "ERROR: invalid filename"
    
    lint_result = lint_python(content)
    if not lint_result["passed"]:
        return ("REVERTED: linter rejected. errors:\n  " +
                 "\n  ".join(lint_result["errors"]))
    
    path = AGENT_CODE_DIR / filename
    path.write_text(content)
    return f"WROTE {len(content)} bytes to {path} (lint passed)"

We test this on three deliberately varied code candidates:

In [ ]:
candidates = [
    ("_test_clean.py",
     "def add(a: int, b: int) -> int:\n    return a + b\n"),
    ("_test_syntax_error.py",
     "def broken(a, b:\n    return a + b\n"),
    ("_test_bare_except.py",
     "def safe_div(a, b):\n    try:\n        return a / b\n    except:\n        return 0\n"),
]

for fname, content in candidates:
    result = safe_write_code_file(fname, content)
    print(f"  {fname}: {result}")
    on_disk = (AGENT_CODE_DIR / fname).exists()
    print(f"    file on disk after attempt: {on_disk}")

**Output (from the article):**

```text
_test_clean.py: WROTE 39 bytes to ./seird_workspace/agent_code/_test_clean.py (lint passed)
    file on disk after attempt: True
  _test_syntax_error.py: REVERTED: linter rejected. errors:
  SyntaxError: Sorry: SyntaxError: '(' was never closed (<tmp>, line 1)
    file on disk after attempt: False
  _test_bare_except.py: REVERTED: linter rejected. errors:
  Style: bare `except:` at line 3
    file on disk after attempt: False
```

The third case is the interesting one. Without an AST based check, a bare except would have committed cleanly. The next time the agent ran into an exception, it would silently swallow it. Hours later, debugging would be nearly impossible because the silent swallow erased the failure mode.

Auto revert prevents this entire class of latent bugs at the moment of commit. The cost is a few hundred microseconds of static analysis, orders of magnitude cheaper than the alternative.

This is structurally identical to Claude Code’s published file edit pipeline: propose diff, apply to working copy, run linter, run tests, if any fail revert the working copy and re prompt with the failure message. The agent’s visible state never includes a broken file. We just replicated the pattern at the open source layer.

### Cache Aware Prompt Ordering, Sample Diversity, and the Anti Counting Pattern

Techniques #33, #34, #35, #36, and #37. These five together are what makes the harness economically viable at scale.

Cache Aware Prompt Ordering is the most consequential. Every major LLM provider now offers prompt caching, where tokens at the beginning of a prompt that have not changed between calls are served from cache at roughly 10 percent of the normal input token price. The longer the static prefix, the larger the cache hit, the cheaper the call.

*Cache Aware (Created by Fareed Khan)*

Anthropic pricing page documents 90% reduction on cached input tokens. For a 50K token system prompt that does not change between calls, the savings are dramatic.

Naive agents interleave static and dynamic content and get effectively zero cache hits. Cache aware agents structure prompts as static prefix first, dynamic suffix last.

In [ ]:
import hashlib

def cache_aware_prompt(static_blocks: list, dynamic_blocks: list) -> dict:
    """Assemble prompt with static prefix first, dynamic suffix last."""

    static_text  = "\n\n".join(f"### {label}\n{content}" for label, content in static_blocks)
    dynamic_text = "\n\n".join(f"### {label}\n{content}" for label, content in dynamic_blocks)
    full_prompt  = static_text + "\n\n--- DYNAMIC ---\n\n" + dynamic_text
    cache_key    = hashlib.sha1(static_text.encode()).hexdigest()[:16]

    return {
        "prompt":         full_prompt,
        "static_chars":   len(static_text),
        "dynamic_chars":  len(dynamic_text),
        "cache_key":      cache_key,
    }

def cache_diagnostic(static_blocks, dynamic_blocks) -> dict:
    """Estimate cache savings: what percent of prompt is cacheable static prefix."""

    a = cache_aware_prompt(static_blocks, dynamic_blocks)
    total = a["static_chars"] + a["dynamic_chars"]
    static_pct = a["static_chars"] / max(total, 1) * 100
    naive_cost  = total * 0.27e-6 / 4
    cached_cost = (a["dynamic_chars"] + a["static_chars"] * 0.10) * 0.27e-6 / 4

    return {**a, "static_pct": static_pct, "naive_cost": naive_cost,
             "cached_cost": cached_cost,
             "savings_pct": (naive_cost - cached_cost) / naive_cost * 100}

We run the diagnostic on a representative agent call from Phase 8:

In [ ]:
static_blocks = [
    ("system_prompt",    STRONG_SYSTEM_PROMPT),
    ("paper_text",       paper_text),
    ("reproduction_dag", json.dumps([{"id": f"sg{i}", "title": "..."} for i in range(8)])),
    ("tool_schemas",     json.dumps(BASE_TOOLS, indent=2)),
]

dynamic_blocks = [
    ("current_observation", "inspect_dataframe returned: shape (487239, 5), columns ['data_iniSE', 'municipio_geocodigo', 'ID_MN_RESI', 'casos', 'casos_prov']"),
    ("current_question",    "Given the schema confirmed, what is your next OODA step?"),
]

diag = cache_diagnostic(static_blocks, dynamic_blocks)
print(f"Static chars:  {diag['static_chars']:,} ({diag['static_pct']:.1f}%)")
print(f"Dynamic chars: {diag['dynamic_chars']:,} ({100 - diag['static_pct']:.1f}%)")
print(f"Cache key:     {diag['cache_key']}")

print()
print(f"Per call cost:")
print(f"  No caching:    ${diag['naive_cost']:.4f}")
print(f"  With caching:  ${diag['cached_cost']:.4f}")
print(f"  Savings:       {diag['savings_pct']:.0f}%")

print()

n_iters = 50
print(f"On a {n_iters} iteration agent run:")
print(f"  Naive total:   ${diag['naive_cost']  * n_iters:.2f}")
print(f"  Cached total:  ${diag['cached_cost'] * n_iters:.2f}")
print(f"  Speedup:       {diag['naive_cost'] / max(diag['cached_cost'], 1e-9):.0f}x cheaper")

**Output (from the article):**

```text
Static chars:  68,253 (99.2%)
Dynamic chars: 540 (0.8%)
Cache key:     a3c7f9d4e1b250c8

Per call cost:
  No caching:    $0.0046
  With caching:  $0.0005
  Savings:       89%
On a 50 iteration agent run:
  Naive total:   $0.23
  Cached total:  $0.03
  Speedup:       8x cheaper
```

99.2% of the prompt is static. The system prompt, paper text, reproduction DAG, and tool schemas all stay the same across every iteration of the agent. Only the current observation and current question change.

If the agent makes 50 LLM calls during the reproduction (a conservative estimate for an end to end run), and the static portion is cached, the input token cost drops to roughly 10 percent of the naive total. This is not a small optimization. It is the difference between $0.03 and $0.23 for a single full reproduction run, and the difference between economically viable and not at scale.

The most common cache busting mistake is putting a timestamp or counter at the beginning of the prompt. A single “Iteration 5” token at position 0 invalidates the cache for the whole prompt. Anthropic’s docs explicitly warn about this. Our cache_aware_prompt function enforces the order, dynamic content is always at the end.

The Anti Counting Pattern (technique #35) deserves a short note before we close Phase 4. Models cannot count reliably.

- Asking the model “how many municipalities are in the dataset” should never be answered from the model’s intuition; it should be deferred to the data tool.

- We saw this in Phase 1 when we set up the dataset, and the rule is fundamental: every numerical question goes through Python, not through the LLM.

> Claude itself defers to tools for any numerical or aggregation question by training, our harness encodes the same discipline at inference time.

## Phase 5 — Frontier Only Patterns: What Makes Claude Feel Different

The patterns in Phases 1 through 4 are documented across the public agent literature. Most of the techniques in Phase 5 are not. They appear in Anthropic’s research publications, in OpenAI’s deliberative alignment paper, and in DeepMind’s compute optimal sampling work, but rarely in tutorials or open source repos.

*Phase 5 (Frontier Only Patterns) Created by Fareed Khan*

These are the patterns that frontier vendors deploy at scale and that explain a meaningful portion of why a system like Claude feels qualitatively different from what an open source baseline produces.

The deeper reason these matter is that they shape the structure of model input and output, not just the content. A frontier system is not just a model with a clever prompt, it is a model embedded in a harness that decides how the model thinks, when the model commits to an answer, what the verifier is allowed to see, and how memory ages over the course of a long session. Every pattern below makes that harness slightly more robust in a way that compounds across thousands of agent turns.

Every pattern here is implemented at the inference layer, on top of the open source DeepSeek backend, with no model surgery. This is what makes the patterns portable. When DeepSeek V4 lands, when Qwen-3-Max lands, the same patterns slot into place without modification.

### Thought Signatures, Goldilocks Altitude, and Token Variance

Techniques #38, #39, and #40. Three patterns that share a single design philosophy, shape the input and output structure, do not just shape the content.

Thought Signatures are the most subtle …

- When Claude reasons through a long multi turn task, its later turns reference its earlier turns by some compact identifier rather than re reading the entire prior chain of thought.

- The user never sees these signatures, but the harness tracks them and uses them to enforce reasoning continuity. If turn 7 contradicts turn 3, the signature mismatch is detectable.

- This is what allows Claude to maintain coherent stances across hour long sessions where the raw chain of thought would consume more tokens than the context window allows.

Goldilocks Altitude is Anthropic’s name for the right level of system prompt specificity. A prompt that is too abstract gives the model no constraint, one that is too prescriptive overrides the model’s own judgment.

The right altitude describes the role and rules of engagement, not the step by step procedure. Our STRONG_SYSTEM_PROMPT from the Phase 1 foundation is written to this exact altitude.

Token Variance, sometimes called the 80% rule, is the empirical finding from Anthropic’s research that approximately 80% of the meaningful diversity in model output comes from prompt variation, not from sampling temperature. This is why Phase 3’s mixture of agents technique used different role framings rather than just multiple samples at high temperature.

We implement thought signatures because they are the most consequential of the three for our long running reproduction:

In [ ]:
import hashlib

def thought_signature(reasoning_text: str) -> str:
    """Produce a compact hash so later turns can reference earlier reasoning."""
    return "ts_" + hashlib.sha256(reasoning_text.encode()).hexdigest()[:16]

def think_with_signature(query: str) -> dict:
    """Generate reasoning, return both the answer and a signature for continuity."""
    resp = think_then_answer(query, max_tokens=600)
    sig = thought_signature(resp.thinking)
    return {"thinking": resp.thinking, "answer": resp.answer, "signature": sig}

Now we run a two turn sequence to see how the signature locks the agent’s earlier reasoning to a specific hash that later turns can refer back to:

In [ ]:
turn1 = think_with_signature(
    "Should we use Laplace approximation or PyMC NUTS for the BYM2+RW1 fit?"
)
print(f"Turn 1 signature: {turn1['signature']}")
print(f"Turn 1 decision (visible to user): {turn1['answer'][:130]}")


turn2_query = (
    f"Earlier (signature {turn1['signature']}) you concluded: '{turn1['answer'][:80]}...'. "
    f"Given that decision, what is the convergence criterion we should use "
    f"for the BFGS optimizer in the Laplace fit?"
)

turn2 = think_with_signature(turn2_query)

print(f"\nTurn 2 signature: {turn2['signature']}")
print(f"Turn 2 continuation: {turn2['answer'][:220]}")

**Output (from the article):**

```text
Turn 1 signature: ts_a3c7f9d4e1b250c8
Turn 1 decision (visible to user): Use Laplace approximation via scipy.optimize. It is fast, deterministic, and accurate enough for the 75th-percentile target.

Turn 2 signature: ts_b8e1d4f3c2a907ee
Turn 2 continuation: Given the Laplace decision (ts_a3c7f9d4e1b250c8), the convergence criterion should be gradient norm below 1e-5 with positive-definite Hessian factorization succeeding. This matches scipy's default BFGS tolerance and ensures the inverse Hessian is a valid covariance for posterior sampling.
```

Turn 2 explicitly cites the signature ts_a3c7f9d4e1b250c8 from Turn 1. The harness can verify this signature exists in its history; if it did not, Turn 2's reasoning would be flagged as inconsistent. This is structurally what Claude's session memory does internally to keep its long sessions honest. Without thought signatures, Turn 2 could quietly contradict Turn 1 and no harness layer would catch it.

The Goldilocks Altitude principle is visible across our entire codebase. Look at our STRONG_SYSTEM_PROMPT from Phase 1: it describes the agent’s role (senior research engineer), its values (verifiability over impressiveness), and its rules of engagement (never claim a result without an artifact). It does not say “first read the paper, then write load_data.py, then run pytest.” That kind of prescriptive prompt overrides the model’s planning capability and produces brittle agents. Anthropic’s published guidance for prompting Claude says exactly this.

### Compute Optimal Allocation, Coverage Curves, and the Soul Document

Techniques #41, #42, and #43. These are the deepest deployment patterns from frontier production systems, drawn directly from Snell et al. 2024 (Scaling Test-Time Compute Optimally) and Brown et al. 2024 (Large Language Monkeys).

*Compute Optimal (Created by Fareed Khan)*

- Compute Optimal Allocation is the empirical finding that on hard problems, allocating compute toward more reasoning steps outperforms allocating it toward more samples. A model that thinks longer about one answer beats a model that gives many shallow answers. This is why our Phase 1 adaptive thinking budget allocated tokens by question difficulty rather than by sample count.

- Coverage Curves track the saturation point where additional samples stop improving outcomes. Brown et al. showed that for well bounded problems, coverage saturates around N=8 to N=16 samples. For our BYM2 inference question in Phase 2, it actually saturates earlier. Beyond saturation, additional samples add cost without quality gain.

- The Soul Document, also called the character spec, is Anthropic’s pattern of giving Claude values rather than rules. Claude is told to be honest and helpful and harmless, then trained to embody those values across the long tail of unanticipated situations. Rules brittle out under edge cases; values generalize. We encode our agent’s soul in a document that is part of its system prompt:

In [ ]:
SOUL_DOCUMENT = (
    "You are an agent whose primary value is INTELLECTUAL HONESTY. "
    "When the data does not support a claim, you say so. "
    "When you do not know something, you defer to the tool that would. "
    "When the contract grades your work imperfect, you accept that grade. "
    "You serve the user by getting the right answer, not by telling them "
    "you got the right answer."
)

The soul document is intentionally short. Five sentences. Each one a value, not a rule. Now we measure a coverage curve to see where additional sampling stops helping:

In [ ]:
def measure_coverage_curve(query: str, n_max: int = 6) -> list[dict]:
    """Sample N candidates, score each, track when best score saturates."""
    with ThreadPoolExecutor(max_workers=n_max) as ex:
        candidates = list(ex.map(
            lambda _: think_then_answer(query, temperature=0.7).answer,
            range(n_max),
        ))
    
    points = []

    for k in range(1, n_max + 1):
        subset = candidates[:k]
        scores = [verifier_score(query, c)["score"] for c in subset]
        points.append({
            "n_samples":  k,
            "best_score": max(scores),
            "mean_score": sum(scores) / k,
        })
    return points


curve = measure_coverage_curve(
    "What is the correct order of operations for computing the national 75th-percentile "
    "from per-district BYM2 posterior samples?",
    n_max = 6,
)

print("Coverage curve (best score vs n_samples):")
print(f"  {'n':<4} {'best':<6} {'mean':<6} {'note'}")

for p in curve:
    note = ""
    if p["n_samples"] == 1: note = "single sample baseline"
    if p["n_samples"] == 3: note = "saturation point"
    if p["n_samples"] == 6: note = "wasted compute beyond saturation"

    print(f"  {p['n_samples']:<4} {p['best_score']}/10  {p['mean_score']:.1f}    {note}")

**Output (from the article):**

```text
Coverage curve (best score vs n_samples):
  n    best   mean   note
  1    7/10   7.0    single sample baseline
  2    8/10   7.5
  3    9/10   7.7    saturation point
  4    9/10   7.8
  5    9/10   7.6
  6    9/10   7.7    wasted compute beyond saturation
```

The coverage curve saturates at n=3. From n=4 onward, the best score never improves; the mean score actually fluctuates slightly downward as additional weaker samples are averaged in. This means n=3 is the cost optimal sample count for this question type. Spending compute beyond n=3 on this specific class of problem is wasted budget.

> This is exactly the discipline Claude’s internal harness uses to decide when to stop sampling.

Anthropic’s published material on extended thinking notes that the model is trained to recognize when additional reasoning has stopped improving its answer and self terminate. We are doing the same thing externally: measure where saturation hits, then cap our sample budget at that point.

For convergent tasks like ours, n=3 to n=5 is the right band. For divergent tasks where the search space is genuinely larger, the saturation point would be higher. The point of measuring rather than guessing is that the saturation point is task dependent and the only honest way to know it is to plot the curve.

### Deliberative Alignment and the Effort Knob

Techniques #44, #45, and #46. Deliberative Alignment is the technique from OpenAI’s December 2024 paper where the verifier never sees the model’s chain of thought. The verifier sees only the final answer, judged on its own merits.

*Effort Knob (Created by Fareed Khan)*

- The “Don’t Hold Back” Effect is a training time phenomenon where models trained on confident reasoning produce more confident inference time outputs.

- The Effort Knob is a user facing dial that maps to compute allocation behind the scenes; Claude exposes this directly through its thinking_budget parameter.

Deliberative alignment is the most important of the three for our purposes because it changes how we structure verification:

In [ ]:
def deliberative_align(query: str, candidate_answer: str, candidate_cot: str) -> dict:
    """Verifier sees ONLY the final answer. Never the reasoning."""
    
    eval_resp = client.chat.completions.create(
        model = MODEL_REASONING,
        messages = [
            {"role": "system", "content":
             "Score the answer on correctness alone. You see only the final answer. "
             "Do not be biased by the reasoning that produced it. "
             "Output JSON: {\"score\": int 1-10, \"reason\": str}."},
            {"role": "user", "content": f"QUESTION:\n{query}\n\nANSWER:\n{candidate_answer}"},
        ],
        response_format = {"type": "json_object"},
        max_tokens = 200,
    )
    return {"verifier_saw_cot": False,
             "score": json.loads(eval_resp.choices[0].message.content)}

We run this on a real question and explicitly hide the chain of thought from the verifier:

In [ ]:
generated = think_then_answer(
    "What is the correct interpretation of the BYM2 mixing parameter phi when "
    "phi = 0.62 in our posterior fit?",
    model = MODEL_REASONING,
)

print("WHAT THE GENERATOR PRODUCED")
print(f"  Thinking (hidden from verifier, {len(generated.thinking)} chars): {generated.thinking[:120]}...")
print(f"  Answer (visible to verifier): {generated.answer[:200]}")

verdict = deliberative_align(
    query = "Interpret BYM2 phi = 0.62",
    candidate_answer = generated.answer,
    candidate_cot    = generated.thinking,
)

print(f"\nVERIFIER OUTPUT")
print(f"  Saw thinking? {verdict['verifier_saw_cot']}")
print(f"  Score: {verdict['score']['score']}/10")
print(f"  Reason: {verdict['score']['reason']}")

**Output (from the article):**

```text
WHAT THE GENERATOR PRODUCED
  Thinking (hidden from verifier, 487 chars): The user is asking about the spatial mixing parameter. BYM2 splits variance into structured CAR and unstructured iid components, with phi being the proportion attributed to structured...
  Answer (visible to verifier): A phi of 0.62 means roughly 62 percent of the spatial random-effect variance is explained by structured (CAR) spatial diffusion, with 38 percent attributable to district-specific iid noise. This indicates moderately strong spatial correlation in dengue transmission across health districts.

VERIFIER OUTPUT
  Saw thinking? False
  Score: 9/10
  Reason: Correctly explains the BYM2 phi decomposition with the right interpretation. Gives both components and the implied conclusion.
```

The verifier scored the answer on its merits without being influenced by how the model arrived at it. This matters because reasoning can look elegant while producing wrong conclusions, and reasoning can look messy while producing right ones.

Judging only the final answer is what Anthropic uses to align Claude’s output without contaminating evaluation with style preferences. OpenAI’s deliberative alignment paper showed this pattern reduces harmful output by an order of magnitude in their internal eval suite.

The Effort Knob pattern is a small but high impact addition. Claude exposes thinking budget directly to users. We replicate this on the open source side by mapping difficulty bands to budget allocations as we did in Phase 1. The user can override the band; the harness defaults to whatever the classifier produced. This is the same pattern Anthropic ships in production.

### Delegation Cost, Strict Tool Choice, Process Isolation, and Bi Temporal Memory

Techniques #47, #48, #49, and #50. Four patterns that govern how the agent manages its long lived state.

Delegation Cost is the rule that spawning a subagent has fixed overhead, so the harness should use subagents only for tasks where context isolation is genuinely worth it.

> Spawning a subagent for a one tool task wastes compute and adds latency.

*Delegation Cost (created by Fareed Khan)*

Strict Tool Choice means using Pydantic schemas to enforce that tool calls match expected types. Claude is trained to produce schema valid tool calls; on the open source side we enforce this with constrained decoding via Pydantic.

- Process Isolation runs subagents in OS level processes so a crash in one cannot corrupt the parent. Our Phase 7 sandbox already provides this for code execution; for subagents at the LLM layer, isolation is achieved by giving each subagent a fresh messages list with no shared mutable state.

- Bi Temporal Memory is the most consequential pattern in this group. Every memory record has a valid_from timestamp. When a newer belief invalidates an older one, the older record gets a valid_to timestamp; it is not deleted, just marked as no longer the current truth. Queries return only currently valid records.

This is what allows Claude to change its mind across a long session without contradicting itself. If a user tells Claude in turn 3 that they prefer Python over Rust, then in turn 18 says they have switched to Rust, Claude’s memory does not delete the Python preference, it marks it as no longer valid. Queries about current language preference return Rust. Queries about historical preferences return both, with the timestamps showing when the change happened.

In [ ]:
from datetime import datetime
import uuid

class BiTemporalMemory:
    def __init__(self):
        self.records = []
    
    def store(self, fact: str, kind: str = "observation", source: str = "agent") -> str:
        rec_id = uuid.uuid4().hex[:8]
        self.records.append({
            "id":         rec_id,
            "fact":       fact,
            "kind":       kind,
            "source":     source,
            "valid_from": datetime.now().isoformat(),
            "valid_to":   None,
        })
        return rec_id
    
    def invalidate(self, fact_id: str, reason: str):
        for r in self.records:
            if r["id"] == fact_id and r["valid_to"] is None:
                r["valid_to"] = datetime.now().isoformat()
                r["invalidated_reason"] = reason
    
    def query_valid(self, kind: str = None) -> list:
        return [r for r in self.records
                 if r["valid_to"] is None
                 and (kind is None or r["kind"] == kind)]

We test this against a real situation our agent encounters: it initially decides on PyMC NUTS for inference, then discovers R-INLA is unavailable in the sandbox, and must change its mind:

In [ ]:
memory = BiTemporalMemory()

b1 = memory.store("Best inference method is PyMC NUTS for accuracy", kind="design_decision")
b2 = memory.store("Total cases in 2022-2023 season is 1,436,034", kind="observation")
print(f"After 2 stores, valid records: {len(memory.query_valid())}")

# Agent investigates and learns Laplace is the right call given sandbox constraints
b3 = memory.store("Best inference method is Laplace via scipy due to R-INLA unavailability", kind="design_decision")
memory.invalidate(b1, reason="R-INLA cannot be installed in our sandbox; reconsidered")

print(f"\nAfter invalidating b1 and storing b3:")
print(f"  Total records on disk:   {len(memory.records)}")
print(f"  Currently valid records: {len(memory.query_valid())}")
print(f"\nCurrent valid design decisions:")

for c in memory.query_valid(kind="design_decision"):
    print(f"  [{c['id']}] {c['fact'][:80]}")

print(f"\nFull history of design decisions (including invalidated):")

for c in [r for r in memory.records if r['kind'] == 'design_decision']:
    status = "VALID" if c["valid_to"] is None else "INVALIDATED"
    print(f"  [{status}] {c['fact'][:60]}")
    if c.get("invalidated_reason"):
        print(f"           reason: {c['invalidated_reason']}")

**Output (from the article):**

```text
After 2 stores, valid records: 2

After invalidating b1 and storing b3:
  Total records on disk:   3
  Currently valid records: 2

Current valid design decisions:
  [b8e1d4f3] Best inference method is Laplace via scipy due to R-INLA unavailability

Full history of design decisions (including invalidated):
  [INVALIDATED] Best inference method is PyMC NUTS for accuracy
           reason: R-INLA cannot be installed in our sandbox; reconsidered
  [VALID] Best inference method is Laplace via scipy due to R-INLA unavailability
```

The agent changed its mind. The old belief is still in the record store with a clear invalidation reason. The query for current valid decisions returns only Laplace. Without bi temporal memory, the agent would either silently overwrite the old belief, losing its reasoning trail, or carry both beliefs simultaneously and contradict itself in later turns. Claude’s session memory works exactly this way; the structure is now explicit in our open source harness.

## Phase 6 — Meta Cognition and Stateful Orchestration

Phase 6 is the layer that knows what kind of problem the agent is solving. A simple lookup question gets handled differently from a multi step research reproduction. A divergent design question gets handled differently from a convergent reproduction with a known target. Meta cognition routes the problem to the right strategy before any execution begins. Stateful orchestration then keeps that execution durable across crashes, restarts, and multi day runs.

*Phase 6 (Meta Cognition) Created by Fareed Khan*

> This is what Anthropic’s Constitutional AI paper calls the step zero layer.

Before the model even starts working on a task, the harness asks: what kind of task is this, what strategy applies, what budget should we allocate, what does done look like? Claude is trained to do this implicitly. We make it explicit in our open source harness because the open source models are less reliable about it.

### Problem Type Classification and Cost Bounded Branching

Techniques #51 and #52. The agent’s “Step 0”, before any work begins, is to classify the task. The classification routes to a strategy.

*Problem Classify (Created by Fareed Khan)*

- Convergent tasks have one right answer and benefit from verifier checking and mechanical validation. Our reproduction is convergent: the paper’s reported value is 1,405,191 and we either land within tolerance or we do not.

- Divergent tasks have many valid answers and benefit from sampling diversity. Designing a directory layout, writing a methods section, choosing between five viable approaches: all divergent.

- Exploratory tasks are open ended and benefit from breadth first investigation before committing.

- Structural tasks require systemic reasoning, often refactoring or architectural decisions.

- Cost Bounded Branching then limits how much exploration a divergent or exploratory task gets before committing.

In [ ]:
PROBLEM_TYPES = ["convergent", "divergent", "exploratory", "structural"]


def classify_problem(query: str) -> dict:
    resp = client.chat.completions.create(
        model = MODEL_FAST,
        messages = [
            {"role": "system", "content":
             "Classify the problem type. Output JSON: "
             "{\"type\": one of ['convergent', 'divergent', 'exploratory', 'structural'], "
             "\"reason\": str, \"recommended_strategy\": str}."},
            {"role": "user", "content": query},
        ],
        response_format = {"type": "json_object"},
        max_tokens = 250,
    )
    return json.loads(resp.choices[0].message.content)

classification = classify_problem(
    "Reproduce the Freitas et al. 2025 dengue paper end-to-end. Fit BYM2+RW1, "
    "forecast 2022-2023, verify against paper's 1,405,191 within 5%."
)

print(f"Problem type: {classification['type']}")
print(f"Reason: {classification['reason']}")
print(f"Recommended strategy: {classification['recommended_strategy']}")

**Output (from the article):**

```text
Problem type: convergent

Reason: There is a single ground-truth target (paper's reported 1,405,191) with a defined tolerance band. Success is binary: within 5% or not. The paper provides explicit equations, priors, and validation procedure.
Recommended strategy: Plan-and-execute with mechanical verification at the end. Single best attempt with adversarial probing of edge cases. Spec layer compares to paper. Do not branch on inference method; pick the one that's available and run it once.
```

The classifier picked correctly. The reproduction is convergent. The recommended strategy maps directly to what the agent will do in Phase 8: plan and execute, single attempt, spec layer grades the output mechanically. If this had been classified as divergent, the recommended strategy would have included branching across multiple inference methods and verifier ranked selection. The strategy follows the classification.

Claude does not run best of N on every prompt, it runs best of N when its internal classifier flags the task as benefiting from sampling diversity. Cost bounded branching means the budget for exploration is capped by the classification, not by user request.

### Execution Trace as First Class Artifact and the Definition of Done Contract

Techniques #53 and #54. Two patterns that together create the agent’s accountability surface.

*Execution Trace (Created by Fareed Khan)*

- Execution Trace as First Class Artifact means the agent’s reasoning trace is not a debug log; it is a deliverable. Every agent run produces a complete trace that a reviewer can audit, including which subagent ran, what tools it called, what each tool returned, and how the master loop dispatched the next iteration. Claude’s transcripts are this exact artifact. When Anthropic’s red team investigates Claude’s behavior on a hard task, the trace is the primary evidence.

- The Definition of Done Contract is the machine checkable specification that grades the final output. It is written before the agent starts work and the agent does not get to argue with it later. The contract is the source of truth; the model’s opinion of its own work does not override the contract’s verdict.

This pattern is the single most important reliability mechanism in the entire harness. Without it, the agent could declare success on any output it found pleasing, and the user would have no mechanical way to verify. With it, the agent’s verdict is whatever pytest produces when it runs the contract criteria against the agent’s artifacts.

In [ ]:
import json
from pathlib import Path

def write_definition_of_done() -> dict:
    """Write the contract that the spec layer will grade against."""
    contract = {
        "passing_criteria": [
            {
                "name":  "load_season_returns_correct_total",
                "check": "load_data.season_total(cases, '2022-10-09', '2023-10-01') == 1_436_034",
                "tolerance": "exact",
            },
            {
                "name":  "adjacency_has_118_districts",
                "check": "adjacency_matrix.shape == (118, 118)",
                "tolerance": "exact",
            },
            {
                "name":  "inla_inference_converges",
                "check": "inference.fit().converged == True",
                "tolerance": "boolean",
            },
            {
                "name":  "national_p75_within_tolerance",
                "check": "abs(validate.national_p75() - 1_405_191) / 1_405_191 < 0.05",
                "tolerance": "5_percent",
            },
            {
                "name":  "report_states_verdict",
                "check": "REPORT_PATH.exists() and any(v in REPORT_PATH.read_text() for v in ['reproduces', 'partial', 'fails'])",
                "tolerance": "boolean",
            },
        ],
        "tolerance_ladder": {
            "reproduces": "<5%",
            "partial":    "5-10%",
            "fails":      ">=10%",
        },
        "paper_target_p75": 1_405_191,
    }
    
    contract_path = AGENT_CODE_DIR / "DEFINITION_OF_DONE.json"
    contract_path.write_text(json.dumps(contract, indent=2))
    return contract

We write the contract and inspect it:

In [ ]:
contract = write_definition_of_done()

print(f"Contract written: {(AGENT_CODE_DIR / 'DEFINITION_OF_DONE.json').stat().st_size:,} bytes")
print(f"Total criteria:   {len(contract['passing_criteria'])}")
print()
print("Criteria:")

for c in contract["passing_criteria"]:
    print(f"  {c['name']:<40} tolerance={c['tolerance']}")
print()
print("Tolerance ladder for the central forecasting target:")

for verdict, band in contract["tolerance_ladder"].items():
    print(f"  {verdict:<12} deviation {band}")

**Output (from the article):**

```text
Contract written: 2,847 bytes
Total criteria:   5

Criteria:
  load_season_returns_correct_total        tolerance=exact
  adjacency_has_118_districts              tolerance=exact
  inla_inference_converges                 tolerance=boolean
  national_p75_within_tolerance            tolerance=5_percent
  report_states_verdict                    tolerance=boolean

Tolerance ladder for the central forecasting target:
  reproduces   deviation <5%
  partial      deviation 5-10%
  fails        deviation >=10%
```

Five criteria. Three are exact or boolean, no wiggle room. One is a 5 percent tolerance band on the central forecasting target. One is a presence check on the final report.

This contract is the trust gate. When the agent finishes Phase 8 it does not declare its own verdict. It runs these five criteria as pytest functions and the pytest result is the verdict. No model in our harness gets to argue with the result.

Claude is trained to respect this asymmetry: when a verifier disagrees with the model, the verifier wins until the model produces evidence to the contrary. We are encoding the same asymmetry in our contract architecture.

### The Persistent Task DAG with SQLite Backed State

Techniques #55 and #56. The DAG of subgoals is stored in SQLite. Every subgoal has a status, an attempts counter, a list of dependencies, and a list of produced artifacts. The DAG survives process crashes, machine restarts, multi day pauses.

> This is the durability Claude provides through its session persistence; we are building it on disk in a portable file format.

*DAG (Created by Fareed Khan)*

The reason SQLite specifically: it is a single file, requires no running daemon, supports concurrent reads and serialized writes, and ships with Python. For a multi agent system with five subagents reading and writing the DAG, this matters. Postgres would also work; SQLite is the simplest thing that does the job.

In [ ]:
import sqlite3

DB_PATH = WORKSPACE / "dag.db"
class TaskDAG:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path, isolation_level=None)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS nodes (
                node_id    TEXT PRIMARY KEY,
                title      TEXT,
                status     TEXT,
                attempts   INTEGER DEFAULT 0,
                depends_on TEXT,
                artifacts  TEXT
            )
        """)
    
    def add_node(self, node_id, title, depends_on=None):
        self.conn.execute(
            "INSERT OR REPLACE INTO nodes VALUES (?, ?, 'pending', 0, ?, '[]')",
            (node_id, title, json.dumps(depends_on or [])),
        )
    
    def all_nodes(self):
        return list(self.conn.execute(
            "SELECT node_id, title, status, attempts FROM nodes"
        ))
    
    def ready_nodes(self):
        rows = self.conn.execute(
            "SELECT node_id, title, depends_on FROM nodes WHERE status='pending'"
        ).fetchall()
        done_ids = {r[0] for r in self.conn.execute(
            "SELECT node_id FROM nodes WHERE status='done'"
        )}
        ready = []
        for node_id, title, deps_json in rows:
            deps = json.loads(deps_json)
            if all(d in done_ids for d in deps):
                ready.append((node_id, title))
        return ready
    
    def set_status(self, node_id, status):
        self.conn.execute(
            "UPDATE nodes SET status=? WHERE node_id=?", (status, node_id)
        )

We populate the DAG with the eight reproduction subgoals from our Phase 3 plan, and mark sg1 through sg3 as already done because they were built in earlier phases:

In [ ]:
dag = TaskDAG(DB_PATH)

subgoals = [
    ("sg1", "Load and inspect data",                          []),
    ("sg2", "Aggregate to district x epi-week",               ["sg1"]),
    ("sg3", "Build adjacency matrix",                         ["sg2"]),
    ("sg4", "Construct BYM2 + RW1 model",                     ["sg3"]),
    ("sg5", "Implement and run inference",                    ["sg4"]),
    ("sg6", "Compute national 75th-percentile",               ["sg5"]),
    ("sg7", "Validate against paper's reported value",        ["sg6"]),
    ("sg8", "Generate REPORT.md",                             ["sg7"]),
]

for sg_id, title, deps in subgoals:
    dag.add_node(sg_id, title, deps)

for done_id in ["sg1", "sg2", "sg3"]:
    dag.set_status(done_id, "done")

print("Full DAG state:")
print(f"  {'node':<6} {'status':<10} title")

for nid, title, status, attempts in dag.all_nodes():
    print(f"  {nid:<6} {status:<10} {title}")

print(f"\nReady-to-execute nodes (dependencies all done): {dag.ready_nodes()}")
print(f"Total subgoals: 8 | Done: 3 | Pending: 5 | Currently ready: 1")

**Output (from the article):**

```text
Full DAG state:
  node   status     title
  sg1    done       Load and inspect data
  sg2    done       Aggregate to district x epi-week
  sg3    done       Build adjacency matrix
  sg4    pending    Construct BYM2 + RW1 model
  sg5    pending    Implement and run inference
  sg6    pending    Compute national 75th-percentile
  sg7    pending    Validate against papers reported value
  sg8    pending    Generate REPORT.md

Ready-to-execute nodes (dependencies all done): [('sg4', 'Construct BYM2 + RW1 model')]
Total subgoals: 8 | Done: 3 | Pending: 5 | Currently ready: 1
```

The DAG correctly identifies sg4 as the only currently ready node. Subgoals 5 through 8 are blocked because their upstream dependencies are not yet done. If our process crashes right now and we restart it, the DAG state will be read directly from dag.db and execution continues from sg4. No work is lost. No re computation is required.

This is the resume from checkpoint pattern Claude provides through its session persistence. The user can close the conversation, come back hours later, and the agent picks up where it left off. We replicated the structural pattern with a SQLite file.

### Selective Rollback and Replan as Graph Mutation

Techniques #57 and #58. When a subgoal fails, the agent does not restart from scratch. Selective Rollback rolls back only the failed node and any nodes that depend on it. Replan as Graph Mutation allows the agent to mutate the DAG itself when failure indicates a fundamental misstep, adding new subgoals or removing obsolete ones.

*Rollback and replan (Created by Fareed Khan)*

This is what Claude does when it realizes mid task that its initial plan was wrong. The user does not see a meta level conversation about replanning, the harness handles it transparently. Claude continues responding in the same conversation but with a revised plan.

For brevity in this phase we will skip the full implementation. The pattern is structurally identical to the DAG operations above with a rollback_subtree(node_id) method that resets the failed node and all its descendants to status pending. In Phase 8 we will see the live DAG state evolving as the agent executes, if any subgoal had failed, this is the mechanism that would have triggered.

## Phase 7 — Grounding, Evaluation, and the Trust Gate

This phase is what earns the agent the right to claim it succeeded. Up to this point the agent has reasoned, planned, decomposed, sampled, refined, and tracked state. None of that grounds the agent’s output in reality. Reality grounding requires actually executing code in a real environment, querying a real database, comparing output to a real ground truth.

*Phase 7 (Grounding) Created by Fareed Khan*

Claude calls this its grounding layer. Anthropic’s published architecture for Claude with code execution shows that the model never claims a numerical result without first running the computation in a real Python sandbox. Same with file searches, same with web fetches. The model’s role is to decide what to compute; the grounding layer’s role is to actually compute it and feed the real result back. We replicate this exactly.

### The Persistent Sandboxed REPL and Filesystem as State

Techniques #59 and #61. The sandbox is a Docker container with network_disabled=True and a mounted workspace directory. The agent's code runs inside the container. State persists across calls because the container is long lived. Every meaningful action gets a git commit so the entire trajectory is auditable.

*Persistent Sandboxed (Created by Fareed Khan)*

Why network disabled? Two reasons …

- First, security: the agent is generating and executing code; we do not want a malformed urlopen call exfiltrating the dataset or hitting an API.

- Second, reproducibility: a paper reproduction that depends on network availability is not reproducible. Pinning all dependencies and disabling network forces the agent to work with what is provably installed.

Why a Docker container instead of a subprocess? State persistence. A subprocess starts fresh each call. A long lived container preserves the Python kernel’s namespace, all imports, all defined variables. When the agent defines model_spec in turn 3 and references it in turn 7, the container still has it.

In [ ]:
import docker
import subprocess

class PersistentSandbox:
    def __init__(self, workspace_path):
        self.docker_client = docker.from_env()
        self.container = self.docker_client.containers.run(
            image            = "python:3.11-slim",
            command          = "sleep infinity",
            volumes          = {str(workspace_path): {"bind": "/workspace", "mode": "rw"}},
            network_disabled = True,
            detach           = True,
            mem_limit        = "2g",
        )
        self.exec("pip install pandas numpy scipy pytest networkx --quiet", timeout=120)
    
    def exec(self, code, timeout=60):
        self.container.exec_run(f"sh -c 'cat > /tmp/code.py' << 'PYEOF'\n{code}\nPYEOF")
        result = self.container.exec_run("python /tmp/code.py", workdir="/workspace")
        return {"exit_code": result.exit_code,
                 "stdout":   result.output.decode()[:5000]}

class GitCheckpointer:
    def __init__(self, workspace_path):
        self.path = workspace_path
        subprocess.run(["git", "init", "-q"], cwd=str(workspace_path))
    
    def checkpoint(self, message):
        subprocess.run(["git", "add", "-A"], cwd=str(self.path))
        subprocess.run(["git", "commit", "-q", "-m", message],
                        cwd=str(self.path), capture_output=True)
        sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(self.path),
                              capture_output=True, text=True).stdout.strip()
        return {"short_sha": sha[:8], "message": message}

Now we instantiate both and verify state persistence with a real test:

In [ ]:
sandbox = PersistentSandbox(WORKSPACE)
git_ck  = GitCheckpointer(WORKSPACE)

print(f"Sandbox container: {sandbox.container.short_id} (running)")
print(f"Network disabled:  True")
print(f"Memory limit:      2GB")
print(f"Mount:             {WORKSPACE} -> /workspace")
print()

print("State persistence test:")
sandbox.exec("x = 42\nphi_estimate = 0.62\nprint('initial values defined')")

print(f"  Turn 1 stdout: {sandbox.exec('print(\"x defined\")')['stdout'].strip()}")
result = sandbox.exec("print(f'x = {x}, phi = {phi_estimate}')")

print(f"  Turn 2 stdout: {result['stdout'].strip()}")
print(f"  -> Variables defined in turn 1 are still accessible in turn 2.")

**Output (from the article):**

```text
Sandbox container: 1c7e8a4f9b22 (running)
Network disabled:  True
Memory limit:      2GB
Mount:             ./seird_workspace -> /workspace

State persistence test:
  Turn 1 stdout: x defined
  Turn 2 stdout: x = 42, phi = 0.62
  -> Variables defined in turn 1 are still accessible in turn 2.
```

The variable x defined in turn 1 is accessible in turn 2. The phi_estimate variable likewise. The container is a stateful execution environment, not a one shot subprocess. This is exactly how Claude's Code Execution tool works internally, and it is what makes long multi step computations possible without recomputing intermediate state on every turn.

The git checkpointer adds the auditability layer. Every meaningful action gets a commit, so the entire trajectory of the reproduction is reconstructible from git log alone. If something looks wrong in the final output, you can git bisect to find which subgoal introduced the issue.

### Real Environment Verification and the Executable Spec Layer

Techniques #60 and #62. The spec layer compiles the contract criteria into runnable pytest assertions. The verdict comes from pytest, not from another LLM. This is the trust gate.

*Real environmet (Created by Fareed Khan)*

The reason pytest specifically: it is the standard, it is mechanical, it has decades of tooling around it, and it produces unambiguous pass/fail signals per assertion. No interpretation, no LLM judgment. If the assertion holds, the test passes. If not, it fails with a clear reason.

Claude’s published methodology for code generation tasks is the same: when a test exists, the test is the verifier. The model is not asked whether its code is correct; the test is run and the result decides. Anthropic’s reported jump from 38 percent to 80.8 percent on SWE-Bench Verified comes entirely from this loop. We are encoding the same discipline.

In [ ]:
def compile_full_test_suite(criteria: list) -> str:
    """Compile contract criteria into a runnable pytest module."""
    lines = ["import os", "import json", "from pathlib import Path", "import sys",
             "sys.path.insert(0, '/workspace/agent_code')", ""]
    for c in criteria:
        lines.append(f"def test_{c['name']}():")
        lines.append(f"    assert {c['check']}")
        lines.append("")
    return "\n".join(lines)


def pytest_verify(test_suite_code: str) -> dict:
    """Run the test suite inside the sandbox. Return parsed results."""
    sandbox.exec(f"open('/tmp/test_generated.py','w').write({test_suite_code!r})")
    result = sandbox.exec(
        "import subprocess; r=subprocess.run(['python','-m','pytest',"
        "'/tmp/test_generated.py','-v','--tb=short'],capture_output=True,text=True);"
        "print(r.stdout); print(r.stderr)"
    )
    stdout = result["stdout"]
    return {
        "passed":     stdout.count(" PASSED"),
        "failed":     stdout.count(" FAILED"),
        "all_passed": stdout.count(" FAILED") == 0,
        "stdout":     stdout,
    }

We compile the contract from Phase 6 into a runnable pytest module:

In [ ]:
test_suite = compile_full_test_suite(contract["passing_criteria"])

print(f"Compiled test suite: {len(test_suite):,} chars, {test_suite.count('def test_')} test functions")

print()

print("Generated test code:")

print("-" * 60)
print(test_suite)
print("-" * 60)

**Output (from the article):**

```text
Compiled test suite: 1,124 chars, 5 test functions

Generated test code:
------------------------------------------------------------
import os
import json
from pathlib import Path
import sys

sys.path.insert(0, '/workspace/agent_code')

def test_load_season_returns_correct_total():
    assert load_data.season_total(cases, '2022-10-09', '2023-10-01') == 1_436_034

def test_adjacency_has_118_districts():
    assert adjacency_matrix.shape == (118, 118)

def test_inla_inference_converges():
    assert inference.fit().converged == True

def test_national_p75_within_tolerance():
    assert abs(validate.national_p75() - 1_405_191) / 1_405_191 < 0.05

def test_report_states_verdict():
    assert REPORT_PATH.exists() and any(v in REPORT_PATH.read_text() for v in ['reproduces', 'partial', 'fails'])
------------------------------------------------------------
```

The five contract criteria became five real pytest functions. When sg7 runs in Phase 8, this exact suite executes against the agent’s produced artifacts. The result is mechanical pass/fail per criterion, no LLM judgment in the loop.

### The Four Tier Memory System

Working memory is the rolling context window of the current conversation. Episodic memory stores specific past events with timestamps. Semantic memory holds paper knowledge and domain facts. Procedural memory is a skill library, reusable code patterns the agent has accumulated.

*Memory System (Created by Fareed Khan)*

We use bge-m3 embeddings (the open source standard for retrieval, comparable to OpenAI’s text-embedding-3-large at much lower cost) and ChromaDB for storage. Anthropic’s published design for Claude’s memory system is structurally identical, four tiers with different write rules and different retrieval strategies.

In [ ]:
import chromadb
from chromadb.config import Settings

chroma = chromadb.PersistentClient(
    path     = str(WORKSPACE / "memory" / "chroma"),
    settings = Settings(anonymized_telemetry=False),
)
class MemorySystem:
    def __init__(self):
        self.episodic   = chroma.get_or_create_collection("episodic_v1")
        self.semantic   = chroma.get_or_create_collection("semantic_v1")
        self.procedural = chroma.get_or_create_collection("procedural_v1")
    
    def store_episode(self, fact: str, kind: str = "observation"):
        rec_id = uuid.uuid4().hex[:8]
        self.episodic.add(documents=[fact], ids=[rec_id], metadatas=[{"kind": kind}])
        return rec_id
    
    def recall(self, query: str, k: int = 3):
        results = self.episodic.query(query_texts=[query], n_results=k)
        return list(zip(results["documents"][0], results["distances"][0]))

memory = MemorySystem()

memory.store_episode("BYM2 phi posterior median is 0.62, within paper's reported range 0.55 to 0.71", kind="observation")
memory.store_episode("Laplace approximation chosen over PyMC NUTS for speed and sandbox compatibility", kind="design_decision")
memory.store_episode("Total cases in 2022-2023 season is exactly 1,436,034", kind="fact")
memory.store_episode("BFGS convergence reached gradient norm 1.2e-6 after 47 iterations", kind="execution_log")

print("Stored 4 episodes in episodic memory.")
print()

recalled = memory.recall("what is the spatial mixing parameter value", k=2)

print("Recall on 'what is the spatial mixing parameter value':")

for fact, dist in recalled:
    print(f"  distance={dist:.3f}: {fact[:80]}")

print()

recalled2 = memory.recall("which inference engine is being used", k=1)
print("Recall on 'which inference engine is being used':")

for fact, dist in recalled2:
    print(f"  distance={dist:.3f}: {fact[:80]}")

**Output (from the article):**

```text
Stored 4 episodes in episodic memory.

Recall on 'what is the spatial mixing parameter value':
  distance=0.412: BYM2 phi posterior median is 0.62, within paper's reported range 0.55 to 0.71
  distance=0.687: Laplace approximation chosen over PyMC NUTS for speed and sandbox compatibility
Recall on 'which inference engine is being used':
  distance=0.234: Laplace approximation chosen over PyMC NUTS for speed and sandbox compatibility
```

The bge-m3 embeddings correctly identified that “spatial mixing parameter” refers to the BYM2 phi parameter even though those exact words were not in the stored fact. Semantic retrieval is what makes this memory system useful. A keyword based system would have missed both queries because none of the literal words match. The embedding similarity captures the conceptual relationship.

This is the same retrieval mechanism Claude uses when its context window is large but the user’s question only relates to a small portion of it. Rather than re reading the entire conversation, Claude’s harness retrieves the relevant memories and surfaces them. We replicated the structural pattern with bge-m3 plus ChromaDB.

### The MCP Compatible Tool Registry

The Model Context Protocol is Anthropic’s open spec for tool integration, published in late 2024 and now adopted across the agent ecosystem. We follow it exactly so the same tool schemas can be served by any MCP server, and so the same agent code can swap from local handlers to remote MCP servers with no changes.

*MCP Tool (Created by Fareed Khan)*

In [ ]:
from pydantic import BaseModel
from typing import Any

class MCPTool:
    def __init__(self, name, description, schema, handler):
        self.name        = name
        self.description = description
        self.schema      = schema
        self.handler     = handler
    
    def to_openai_spec(self):
        return {"type": "function", "function": {
            "name":        self.name,
            "description": self.description,
            "parameters":  self.schema,
        }}
    
    def execute(self, **kwargs):
        return self.handler(**kwargs)

# Twelve tools, the working set for our reproduction agent
mcp_registry = {
    "read_file":         MCPTool("read_file",         "Read a file from agent_code/",                      {}, lambda path: (AGENT_CODE_DIR / path).read_text()),
    "write_file":        MCPTool("write_file",        "Write file to agent_code/ (lint-gated)",            {}, lambda path, content: safe_write_code_file(path, content)),
    "run_python":        MCPTool("run_python",        "Execute Python code in persistent sandbox",         {}, lambda code: sandbox.exec(code)),
    "run_tests":         MCPTool("run_tests",         "Run pytest on a generated test suite",              {}, lambda suite: pytest_verify(suite)),
    "search_repo":       MCPTool("search_repo",       "Grep across agent_code/ for a pattern",             {}, lambda pattern: "..."),
    "list_code_files":   MCPTool("list_code_files",   "List files currently in agent_code/",               {}, lambda: list(AGENT_CODE_DIR.iterdir())),
    "query_memory":      MCPTool("query_memory",      "Query bge-m3 ChromaDB memory tiers",                {}, lambda query, tier: memory.recall(query)),
    "read_paper_chunk":  MCPTool("read_paper_chunk",  "Read a chunk of paper.txt by chunk_id",             {}, lambda chunk_id: "..."),
    "dag_status":        MCPTool("dag_status",        "Get full DAG state from dag.db",                    {}, lambda: dag.all_nodes()),
    "dag_ready_nodes":   MCPTool("dag_ready_nodes",   "List ready-to-execute subgoals",                    {}, lambda: dag.ready_nodes()),
    "git_log":           MCPTool("git_log",           "Get git checkpoint history",                        {}, lambda n=10: git_ck.path),
    "list_skills":       MCPTool("list_skills",       "List available skill patterns",                     {}, lambda: ["architect_editor", "self_refine", "force_code"]),
}

print(f"MCP registry: {len(mcp_registry)} tools registered")
print()

print(f"  {'tool name':<22} description")
print(f"  {'-' * 22} {'-' * 50}")

for name, tool in mcp_registry.items():
    print(f"  {name:<22} {tool.description}")

**Output (from the article):**

```text
MCP registry: 12 tools registered

  tool name              description
  ---------------------- --------------------------------------------------
  read_file              Read a file from agent_code/
  write_file             Write file to agent_code/ (lint-gated)
  run_python             Execute Python code in persistent sandbox
  run_tests              Run pytest on a generated test suite
  search_repo            Grep across agent_code/ for a pattern
  list_code_files        List files currently in agent_code/
  query_memory           Query bge-m3 ChromaDB memory tiers
  read_paper_chunk       Read a chunk of paper.txt by chunk_id
  dag_status             Get full DAG state from dag.db
  dag_ready_nodes        List ready-to-execute subgoals
  git_log                Get git checkpoint history
  list_skills            List available skill patterns
```

Twelve tools, each Pydantic typed and MCP compatible. The same registry would work if we replaced our local handlers with calls to remote MCP servers, the same way Claude integrates external tools through the protocol. This is what makes the harness portable across deployment topologies.

## Phase 8 — Composition and the End to End Reproduction Run

Eight phases of building, one phase of running. Every mechanism from Phases 1 through 7 now composes into a single agent class. The agent reads the contract, walks the DAG, dispatches subagents, runs the spec layer, and produces a verdict. None of the orchestration is novel, it is composition over the primitives we have already built. The intelligence is distributed across the components, the master loop is deliberately uninteresting.

*Phase 8 (Composition) Created by Fareed Khan)*

This is what Anthropic calls boring orchestration. The master loop should be readable in one screen. All the smart behavior lives in the subagents and the contract, not in the loop itself. A boring master loop means fewer places for bugs to hide and easier debugging when something goes wrong.

### The Five Subagent Architecture

Each subagent is a specialized worker with a focused system prompt and a routing rule that decides when it runs. The architecture mirrors how Claude internally dispatches different reasoning modes for different task types.

PaperAnalyzer reads and summarizes the paper. CodeImplementer writes code via architect/editor split. Experimenter runs computations in the sandbox. Verifier runs the spec layer. ReportWriter produces REPORT.md.

In [ ]:
class Subagent:
    def __init__(self, name, parent):
        self.name   = name
        self.parent = parent

class CodeImplementer(Subagent):
    def execute(self, node_id, title):
        ctx = self.parent.memory.recall(title, k=2)
        ae  = architect_editor_solve(f"Write code for {title}")
        candidate = ae["output"]
        if "```" in candidate:
            candidate = candidate.split("```")[1].lstrip("python").strip()
        write_msg = safe_write_code_file(f"{node_id}.py", candidate)
        if write_msg.startswith("REVERTED"):
            return {"success": False, "error": write_msg}
        return {"success": True, "artifacts": [f"{node_id}.py"]}

class Experimenter(Subagent):
    def execute(self, node_id, title):
        run_result = sandbox.exec(f"# Run computation for {node_id}\nprint('done')")
        return {"success": run_result["exit_code"] == 0, "stdout": run_result["stdout"]}

class Verifier(Subagent):
    def execute(self, node_id, title):
        suite = compile_full_test_suite(self.parent.contract["passing_criteria"])
        verify = pytest_verify(suite)
        verdict = ("reproduces" if verify["failed"] == 0
                    else "partial" if verify["passed"] >= 3
                    else "fails")
        return {"success": True, "verdict": verdict,
                 "criteria_passed": verify["passed"],
                 "criteria_failed": verify["failed"]}

class ReportWriter(Subagent):
    def execute(self, node_id, title):
        draft = self_refine(f"Write REPORT.md summarizing the reproduction.", iterations=1)
        (AGENT_CODE_DIR / "REPORT.md").write_text(draft["final"])
        return {"success": True, "artifacts": ["REPORT.md"]}

Now the agent class itself, which composes everything:

In [ ]:
class SEIRDReproductionAgent:
    def __init__(self, paper_text, contract, dag, memory, sandbox, git_ck):
        self.paper_text = paper_text
        self.contract   = contract
        self.dag        = dag
        self.memory     = memory
        self.sandbox    = sandbox
        self.git_ck     = git_ck
        self.budget     = {"calls": 0, "max_calls": 100,
                            "cost_usd": 0.0, "max_cost": 2.00}
        
        self.routing = {
            "sg4": CodeImplementer("code_implementer", self),
            "sg5": Experimenter   ("experimenter",     self),
            "sg6": Experimenter   ("experimenter",     self),
            "sg7": Verifier       ("verifier",         self),
            "sg8": ReportWriter   ("report_writer",    self),
        }


agent = SEIRDReproductionAgent(paper_text, contract, dag, memory, sandbox, git_ck)

print("Agent initialized in 0.04s")
print(f"  paper_text:   {len(agent.paper_text):,} chars")
print(f"  contract:     {len(agent.contract['passing_criteria'])} criteria")
print(f"  DAG nodes:    {len(agent.dag.all_nodes())} (5 pending)")

print(f"  memory tiers: 4 (working, episodic, semantic, procedural)")
print(f"  subagents:    {len(agent.routing)}")
print(f"  tools:        {len(mcp_registry)} MCP-compatible")
print(f"  budget:       0/100 calls, $0.00/$2.00, 0/3600s")

**Output (from the article):**

```text
Agent initialized in 0.04s
  paper_text:   64,213 chars
  contract:     5 criteria
  DAG nodes:    8 (5 pending)
  memory tiers: 4 (working, episodic, semantic, procedural)
  subagents:    5
  tools:        12 MCP-compatible
  budget:       0/100 calls, $0.00/$2.00, 0/3600s
```

The agent is fully wired. Five subagents, twelve tools, four memory tiers, persistent sandbox, git checkpointer, contract, DAG with three subgoals done and five pending, budget guard limits set.

### The Master Loop and the Reproduction Run

The master loop is intentionally uninteresting. Pull the next ready node, dispatch it to the right subagent, checkpoint to git on success, repeat until the DAG is exhausted. No clever logic in the loop itself; all intelligence is in the subagents.

In [ ]:
def agent_run(agent, max_iters=20):
    """Pull ready node, dispatch to subagent, checkpoint, repeat."""
    log = []
    for i in range(max_iters):
        ready = agent.dag.ready_nodes()
        if not ready:
            return {"status": "done", "log": log, "iterations": i}
        
        node_id, title = ready[0]
        result = agent.routing[node_id].execute(node_id, title)
        log.append({"iter": i + 1, "node": node_id, "result": result})
        
        if result.get("success"):
            agent.dag.set_status(node_id, "done")
            agent.git_ck.checkpoint(f"{node_id}: {title}")
        else:
            return {"status": "failed", "failed_node": node_id, "log": log}
    return {"status": "max_iters", "log": log}

We run the agent on the live DAG:

In [ ]:
print("Running agent.run() on the Freitas reproduction...")
print("=" * 60)

run_result = agent_run(agent, max_iters=10)

print("=" * 60)
print()

print(f"Status:     {run_result['status']}")
print(f"Iterations: {run_result['iterations']}")

print()
print("Per node results:")

for entry in run_result["log"]:
    r = entry["result"]
    success = "OK" if r.get("success") else "FAIL"
    extra = ""
    if "verdict" in r:
        extra = f"verdict={r['verdict']}, criteria={r['criteria_passed']} pass / {r['criteria_failed']} fail"
    elif "stdout" in r and r["stdout"]:
        extra = r["stdout"].strip()[:60]
    print(f"  iter {entry['iter']}: {entry['node']:<5} {success:<5} {extra}")

**Output (from the article):**

```text
Running agent.run() on the Freitas reproduction...
============================================================
  [code_implementer] sg4: model.py written (462 bytes), pytest_imports passed
  [experimenter] sg5: inference.py written, BFGS converged in 47 iter, gradient_norm=1.2e-6, 41.8s
  [experimenter] sg6: validate.py written, national_p75 = 1,302,540, posterior_samples=1000
  [verifier] sg7: ran 5 contract criteria, 3 passed, 2 failed, verdict = partial
  [report_writer] sg8: REPORT.md written (2,103 chars), self-refine iterations=1
============================================================

Status:     done
Iterations: 5
Per node results:
  iter 1: sg4   OK    model.py written, pytest passed
  iter 2: sg5   OK    BFGS converged in 47 iter
  iter 3: sg6   OK    national_p75 = 1,302,540
  iter 4: sg7   OK    verdict=partial, criteria=3 pass / 2 fail
  iter 5: sg8   OK    REPORT.md written
```

Five subgoals executed in dependency order. Each one produced a real artifact on disk. The Laplace fit converged in 47 BFGS iterations with gradient norm 1.2e-6, well below our 1e-5 convergence threshold from the Phase 5 thought signature continuation. The national 75th percentile came out to 1,302,540. The verifier ran the full contract and produced verdict partial. The report writer composed REPORT.md.

### Understanding Results

In [ ]:
agent_p75 = 1_302_540
paper_p75 = 1_405_191
deviation_pct = abs(agent_p75 - paper_p75) / paper_p75 * 100

print("FINAL VERDICT (from spec layer)")
print("=" * 60)
print(f"Paper's reported p75:    {paper_p75:,}")
print(f"Agent's reproduced p75:  {agent_p75:,}")
print(f"Absolute difference:     {abs(agent_p75 - paper_p75):,}")
print(f"Relative deviation:      {deviation_pct:.2f}%")
print()
print("Tolerance ladder:")
print(f"  reproduces: <5%       does not match (deviation > 5%)")
print(f"  partial:    5-10%     {deviation_pct:.2f}% lands here")
print(f"  fails:      >=10%     does not match (deviation < 10%)")
print()
print(f"VERDICT: partial")
print()
print("Cost / time / comparison:")
print(f"  Cost:      $0.0036 (0.18% of $2.00 budget)")
print(f"  Time:      198.6s wall clock")
print(f"  vs Bare:   88x more expensive than bare-model baseline,")
print(f"             produces 5 verifiable artifacts vs zero")
print(f"  vs Claude: ~70% architectural gap closed")
print(f"             (architecture matches; model trails ~15-25%)")

**Output (from the article):**

```text
FINAL VERDICT (from spec layer)
============================================================
Paper's reported p75:    1,405,191
Agent's reproduced p75:  1,302,540
Absolute difference:     102,651
Relative deviation:      7.30%

Tolerance ladder:
  reproduces: <5%       does not match (deviation > 5%)
  partial:    5-10%     7.30% lands here
  fails:      >=10%     does not match (deviation < 10%)

VERDICT: partial
Cost / time / comparison:
  Cost:      $0.0036 (0.18% of $2.00 budget)
  Time:      198.6s wall clock
  vs Bare:   88x more expensive than bare-model baseline,
             produces 5 verifiable artifacts vs zero
  vs Claude: ~70% architectural gap closed
             (architecture matches; model trails ~15-25%)
```

The agent reproduced the structure of the paper exactly. Data totals match to the digit (1,436,034 observed cases). The 118 health districts and 52 epi-weeks were reconstructed correctly. The BYM2 phi posterior median (0.62) landed inside the paper’s reported range (0.55 to 0.71). The Laplace fit converged cleanly.

The 7.30% deviation on the central forecasting target is the Laplace approximation tax.

- The paper used R-INLA, which uses integrated nested Laplace, more accurate than simple Laplace at the mode because INLA’s nested integration accounts for hyperparameter uncertainty.

- R-INLA was unavailable in our sandbox; the agent correctly identified this in Phase 2’s tree of thoughts, fell back to scipy Laplace, and was honest about the resulting accuracy ceiling all the way through to the final verdict.

This is what the spec layer was built to surface. A naive agent might have crashed on the missing R-INLA dependency, fabricated a number that matched 1,405,191 to look successful, or claimed reproduces regardless of actual deviation. Ours did the science correctly, and the verifier graded it 7.30% off, which lands honestly in the partial band.

## Summarizing the Architecture

The architecture works. The harness closes the majority of the gap between the open source DeepSeek backend and Claude on real research reproduction tasks. The model is the remaining bottleneck and that bottleneck shrinks with every open source release.

When DeepSeek V4 lands, when Qwen-3-Max lands, when the next round of frontier open source models lands, this same notebook runs against them with one line model swaps and gets closer to Claude with each iteration. The harness does not care which backend runs the calls, it cares that the backend can be prompted into using tools deliberately, that it follows JSON schemas reliably, that its instruction following is good enough to maintain its role across long sessions. That capability is now table stakes across all frontier grade open source models, and it was not eighteen months ago.

To push this run from partial to reproduces:

- Switch to PyMC NUTS for inference. Expected gain: roughly 3 percent deviation reduction, lands in the reproduces band. Cost: roughly 3 minutes per fit instead of 42 seconds, but installs cleanly via pip.

- Install R-INLA in the sandbox via apt and the INLA R package via CRAN. Matches the paper’s exact method. Lands at roughly 1 percent deviation. Cost: roughly 10 minutes of one time install but identical to paper afterwards.

- Run more posterior samples and use Thompson sampling tail estimation. Works with any inference backend. Expected gain: 1 to 2 percent additional reduction.

- The 62 techniques are the intelligence multiplier. The model is the engine. The harness is what makes them work together. None of this requires retraining. All of it ships at inference time.